# ==========================================================================================
# Phase 4 - BTUMQA QGCA QAdp-PRUGTM Hybrid Clean-Metadata Four-Seeds Training (Single Stage)
# BTUMQA-225K Standalone Colab Notebook - Four Seeds
# ==========================================================================================


# Dual Environment Compatibility Setup & Install Required Libraries


In [ ]:
# ── DUAL ENVIRONMENT COMPATIBILITY & DEPENDENCY SETUP ────────────────────────
import os
import sys
from pathlib import Path

def resolve_project_environment(mount_point: str = "/content/drive") -> tuple[Path, Path]:
    try:
        import google.colab
        from google.colab import drive
        drive.mount(mount_point)
        project_root = Path(mount_point) / "MyDrive" / "AUGR-VQA"
        temp_dir = Path("/content")
        print("Running in Google Colab environment.")
    except ImportError:
        # Running locally (parent of notebooks directory)
        project_root = Path(os.getcwd()).parent.resolve()
        temp_dir = project_root / "temp"
        temp_dir.mkdir(parents=True, exist_ok=True)
        print("Running in Local environment.")
    return project_root, temp_dir

PROJECT_ROOT, TEMP_DIR = resolve_project_environment()
# ─────────────────────────────────────────────────────────────────────────────

!pip install -q pandas scikit-learn tqdm

import csv
import gc
import json
import math
import os
import random
import shutil
import time
from collections import Counter, defaultdict
from contextlib import nullcontext
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score
from torch.cuda.amp import GradScaler, autocast
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Torch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


# Mount Google Drive and Phase 4 QA-PRUGTM Hybrid Paths Setup


In [ ]:
from google.colab import drive
from pathlib import Path
import os
import time


def ensure_drive_connection(project_dir: Path, mount_point: str = "/content/drive"):
    def probe_path(target: Path):
        probe_target = target if target.exists() else target.parent
        return os.listdir(str(probe_target))

    try:
        probe_path(project_dir)
    except OSError as exc:
        if getattr(exc, "errno", None) != 107:
            raise
        print("Detected a stale Google Drive mount. Remounting now...")
        try:
            drive.flush_and_unmount()
            time.sleep(2)
        except Exception:
            pass
#         drive.mount(mount_point, force_remount=True)
        time.sleep(2)
        probe_path(project_dir)

    if not project_dir.exists():
        raise FileNotFoundError(
            f"Project Drive directory not found after mount check: {project_dir}"
        )


# drive.mount("/content/drive")

# Updated project path to point to the new drive structure
PROJECT_DRIVE_DIR = PROJECT_ROOT

ensure_drive_connection(PROJECT_DRIVE_DIR)

DATASET_RELEASE_NAME = "BTUMQA-225K"
PHASE3A_RELEASE_KEY = "dataset_btumqa_225k"
PHASE4_MODEL_VARIANT = "btumqa_225k_qgca_qadp_prugtm_hybrid_adaptive_uncertainty_clean_metadata"
PHASE4_METHOD_NAME = "BrainTumorVQA-BTUMQA-225K-QGCA-QAdp-PRUGTM-Hybrid-Clean"
CLEAN_VISUAL_TOKEN_SOURCE = "original_visual_tokens_plus_question_adaptive_region_aux"
PHASE4_FUSION_TYPE = (
    "question-region adaptive uncertainty gating over visual-derived region_aux "
    "with QGCA attention and post-reasoning uncertainty gating"
)

# Corrected paths: phase_2/p2c_ugtm_modulated_tokens
PHASE2C_BASE_DIR = PROJECT_DRIVE_DIR / "phase_2" / "p2c_ugtm_modulated_tokens"
PHASE2C_SAVE_DIR = PHASE2C_BASE_DIR / "modulated_tokens_by_patient"

# Corrected paths: phase_3/p3a_brats_vqa_dataset
PHASE3A_BASE_DIR = PROJECT_DRIVE_DIR / "phase_3" / "p3a_brats_vqa_dataset"
# Corrected paths: phase_3/p3b_text_preprocessing
PHASE3B_BASE_DIR = PROJECT_DRIVE_DIR / "phase_3" / "p3b_text_preprocessing"

# Corrected paths: phase_4/p4a_qgca_training
PHASE4_BASE_DIR = PROJECT_DRIVE_DIR / "phase_4" / "p4a_qgca_training"
PHASE4_LEGACY_PARENT_DIR = PHASE4_BASE_DIR / "single_stage_btumqa_225k_prugtm_hybrid_45e"
PHASE4_CLEAN_METADATA_PARENT_DIR = PHASE4_BASE_DIR / "clean_metadata_training"
PHASE4_PARENT_DIR = PHASE4_CLEAN_METADATA_PARENT_DIR / "adaptive_prugtm_qgca_btumqa_four_seeds"

PHASE3A_DIR = PHASE3A_BASE_DIR / "dataset_btumqa_225k"
PHASE3B_DIR = PHASE3B_BASE_DIR / "dataset_btumqa_225k"
PHASE4_CLEAN_METADATA_PARENT_DIR.mkdir(parents=True, exist_ok=True)
PHASE4_PARENT_DIR.mkdir(parents=True, exist_ok=True)

# Reuse the previous pure post-reasoning QGCA training visual cache.
VISUAL_CACHE_DIR = PHASE4_BASE_DIR / "single_stage_btumqa_225k_prugtm_45e" / "visual_cache"
VISUAL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
VISUAL_CACHE_TENSOR_PATH = VISUAL_CACHE_DIR / "visual_slice_cache_original_tokens.pt"
VISUAL_CACHE_METADATA_PATH = VISUAL_CACHE_DIR / "visual_slice_cache_original_tokens_metadata.json"

SPLIT_PREFIX = "btumqa_225k"
EXPECTED_SPLIT_ROWS = {
    "train": 157500,
    "val": 33750,
    "test": 33750,
}

TRAIN_CSV_PATH = PHASE3A_DIR / f"{SPLIT_PREFIX}_train.csv"
VAL_CSV_PATH = PHASE3A_DIR / f"{SPLIT_PREFIX}_val.csv"
TEST_CSV_PATH = PHASE3A_DIR / f"{SPLIT_PREFIX}_test.csv"

EMBEDDINGS_TRAIN_PATH = PHASE3B_DIR / "text_embeddings_train.pt"
EMBEDDINGS_VAL_PATH = PHASE3B_DIR / "text_embeddings_val.pt"
EMBEDDINGS_TEST_PATH = PHASE3B_DIR / "text_embeddings_test.pt"

ANSWER_VOCAB_PATH = PHASE3B_DIR / "answer_vocab.json"
QUESTION_FAMILY_VOCAB_PATH = PHASE3B_DIR / "question_family_vocab.json"
QUESTION_STYLE_VOCAB_PATH = PHASE3B_DIR / "question_style_vocab.json"
DIFFICULTY_LEVEL_VOCAB_PATH = PHASE3B_DIR / "difficulty_level_vocab.json"
AMBIGUITY_FLAG_VOCAB_PATH = PHASE3B_DIR / "ambiguity_flag_vocab.json"
SIGNAL_GAP_BUCKET_VOCAB_PATH = PHASE3B_DIR / "signal_gap_bucket_vocab.json"
REGION_TARGET_VOCAB_PATH = PHASE3B_DIR / "region_target_vocab.json"
PHASE3B_CONFIG_PATH = PHASE3B_DIR / "phase3b_text_config.json"
PHASE3B_REPORT_PATH = PHASE3B_DIR / "phase3b_text_preprocessing_report.json"

print("Project directory:", PROJECT_DRIVE_DIR)
print("Dataset release:", DATASET_RELEASE_NAME)
print("Phase 3A dir:", PHASE3A_DIR)
print("Phase 3B dir:", PHASE3B_DIR)
print("Phase 4 legacy parent dir:", PHASE4_LEGACY_PARENT_DIR)
print("Bias-strict metadata-free parent dir:", PHASE4_CLEAN_METADATA_PARENT_DIR)
print("Phase 4 parent dir:", PHASE4_PARENT_DIR)
print("Phase 2C save dir:", PHASE2C_SAVE_DIR)
print("Visual cache dir:", VISUAL_CACHE_DIR)
print("Model variant:", PHASE4_MODEL_VARIANT)
print("Method name:", PHASE4_METHOD_NAME)
print("Fusion type:", PHASE4_FUSION_TYPE)
print("Clean visual token source:", CLEAN_VISUAL_TOKEN_SOURCE)
print("Expected rows:", EXPECTED_SPLIT_ROWS)
print()
print("Train CSV exists:", TRAIN_CSV_PATH.exists(), TRAIN_CSV_PATH)
print("Val CSV exists:", VAL_CSV_PATH.exists(), VAL_CSV_PATH)
print("Test CSV exists:", TEST_CSV_PATH.exists(), TEST_CSV_PATH)
print("Train embeddings exist:", EMBEDDINGS_TRAIN_PATH.exists(), EMBEDDINGS_TRAIN_PATH)
print("Val embeddings exist:", EMBEDDINGS_VAL_PATH.exists(), EMBEDDINGS_VAL_PATH)
print("Test embeddings exist:", EMBEDDINGS_TEST_PATH.exists(), EMBEDDINGS_TEST_PATH)
print("Answer vocab exists:", ANSWER_VOCAB_PATH.exists(), ANSWER_VOCAB_PATH)
print("Question family vocab exists:", QUESTION_FAMILY_VOCAB_PATH.exists(), QUESTION_FAMILY_VOCAB_PATH)
print("Question style vocab exists:", QUESTION_STYLE_VOCAB_PATH.exists(), QUESTION_STYLE_VOCAB_PATH)
print("Ambiguity vocab exists:", AMBIGUITY_FLAG_VOCAB_PATH.exists(), AMBIGUITY_FLAG_VOCAB_PATH)
print("Signal gap vocab exists:", SIGNAL_GAP_BUCKET_VOCAB_PATH.exists(), SIGNAL_GAP_BUCKET_VOCAB_PATH)
print("Region target vocab exists:", REGION_TARGET_VOCAB_PATH.exists(), REGION_TARGET_VOCAB_PATH)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project directory: /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging
Dataset release: BTUMQA-225K
Phase 3A dir: /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_3a_brats_vqa_dataset/dataset_btumqa_225k
Phase 3B dir: /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_3b_text_preprocessing/dataset_btumqa_225k
Phase 4 legacy parent dir: /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_4_qgca_training/single_stage_btumqa_225k_prugtm_hybrid_45e
Bias-strict metadata-free parent dir: /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_4_qgca_training/biasless_metadata_free_training
Phase 

# Phase 4 Configuration Setup


In [ ]:
# Colab-free-GPU friendly: run one seed at a time if needed, e.g. RUN_SEEDS = [42].
# Re-running the same seed is safe with RESUME_TRAINING=True; it resumes from last_* checkpoint.
RUN_SEEDS = [42, 1337, 2025, 3407]
LEGACY_SEEDS_TO_COPY = []
TRAINABLE_SEEDS = RUN_SEEDS
SEED = RUN_SEEDS[0]
SEED_TAG = f"seed_{SEED}"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

FIXED_ANSWER_ORDER = [
    "ambiguous",
    "close_gap",
    "confident_present",
    "consistent",
    "context",
    "distinct",
    "edema",
    "enhancing",
    "global",
    "large_confident",
    "large_uncertain",
    "moderate_confident",
    "moderate_gap",
    "moderate_uncertain",
    "ncr_net",
    "none",
    "not_present",
    "shifted",
    "small_confident",
    "small_uncertain",
    "tumor",
    "uncertain_present",
    "wide_gap",
]

FIXED_QUESTION_FAMILY_ORDER = [
    "ambiguous_subregion_pair",
    "confidence_qualified_extent",
    "confidence_qualified_presence",
    "dominant_region_under_uncertainty",
    "highest_risk_region",
    "lowest_risk_region",
    "more_reliable_region",
    "more_uncertain_region",
    "reliability_gap_bucket",
    "safe_region_for_reasoning",
    "uncertainty_consistency_check",
    "uncertainty_gap_bucket",
]

FIXED_QUESTION_STYLE_ORDER = [
    "ambiguity_sensitive",
    "bucketed",
    "comparative",
    "confidence_qualified",
    "ranking",
]

FIXED_DIFFICULTY_LEVEL_ORDER = ["easy", "medium", "hard"]
FIXED_AMBIGUITY_FLAG_ORDER = ["no", "yes"]
FIXED_SIGNAL_GAP_BUCKET_ORDER = ["close_gap", "moderate_gap", "wide_gap"]
FIXED_REGION_TARGET_ORDER = ["", "edema", "ncr_net", "enhancing", "tumor", "context", "global"]

SELECTED_CONTEXT_FIELDS = []
AUDIT_REPORTING_FIELDS = [
    "question_family",
    "question_style",
    "difficulty_level",
    "ambiguity_flag",
    "signal_gap_bucket",
    "region_target_primary",
    "region_target_secondary",
]

CLEAN_MODEL_INPUT_FIELDS = [
    "question_embedding",
    "visual_tokens",
    "region_aux",
]
CLEAN_LABEL_FIELDS = ["answer_id"]
ALIGNMENT_ONLY_FIELDS = ["qa_id", "split", "patient_id", "slice_id", "unique_id", "image_id"]
VISUAL_ARTIFACT_POINTER_FIELDS = [
    "phase2a_token_file",
    "slice_index_in_token_file",
    "phase2c_file",
    "slice_index_in_ugtm_file",
]
VISUAL_DERIVED_TENSOR_INPUTS = [
    "visual_tokens",
    "region_aux.uncertainty",
    "region_aux.modulation_weight",
    "region_aux.region_present",
    "region_aux.relative_area",
]
CHECKPOINT_SAFETY_NOTE = (
    "Each seed writes best_* and last_* checkpoints to Google Drive after every completed epoch. "
    "With RESUME_TRAINING=True, rerunning the same seed resumes from last_* instead of starting over."
)
FORBIDDEN_MODEL_FORWARD_KWARGS = [
    "question_style_id",
    "difficulty_level_id",
    "ambiguity_flag_id",
    "signal_gap_bucket_id",
    "region_target_primary_id",
    "region_target_secondary_id",
]
EXCLUDED_AUDIT_ONLY_FIELDS = [
    "question_family",
    "question_style",
    "template_id",
    "paraphrase_id",
    "difficulty_level",
    "ambiguity_flag",
    "region_target_primary",
    "region_target_secondary",
    "signal_gap_bucket",
    "decision_rule_id",
    "label_provenance",
    "candidate_keep_reason",
    "tumor_slice_csv",
    "raw_tabular_area_uncertainty_weight_columns",
    "mc_dropout_passes",
]

REGION_NAMES = [
    "edema",
    "ncr_net",
    "enhancing",
    "tumor",
    "context",
    "global",
]

TEXT_EMBED_KEY = "cls_last_hidden_state"
TEXT_EMBED_DIM = 768
VISUAL_TOKEN_DIM = 512
MODEL_DIM = 512
AUX_FEATURE_DIM = 4
NUM_VISUAL_TOKENS = 6

NUM_ATTENTION_HEADS = 8
AUX_HIDDEN_DIM = 64
CLASSIFIER_HIDDEN_DIM = 512
CONTEXT_EMBED_DIM = 32
CONTEXT_HIDDEN_DIM = 128
DROPOUT = 0.20
AUX_GATE_INIT_PROB = 0.875

TRAIN_BATCH_SIZE = 96
EVAL_BATCH_SIZE = 192
NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()

MAX_EPOCHS = 50
MIN_EPOCHS_BEFORE_EARLY_STOP = 5
EARLY_STOPPING_PATIENCE = 8

LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.05
GRAD_CLIP_NORM = 1.0
USE_CLASS_WEIGHTS = True

USE_AMP = torch.cuda.is_available()
RESUME_TRAINING = True # Set False when run from start again and False only for first run

STRICT_PHASE2C_ALIGNMENT = True
VISUAL_CACHE_DTYPE = torch.float16
DEBUG_MAX_ROWS_PER_SPLIT = None

REUSE_SAVED_VISUAL_CACHE = True
FORCE_REBUILD_VISUAL_CACHE = False
SKIP_TRAIN_IF_ALREADY_DONE = True
SKIP_FINAL_EVAL_IF_ALREADY_DONE = True
FORCE_RERUN_FINAL_EVAL = False

if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")

print("Run seeds:", RUN_SEEDS)
print("Legacy seeds to copy:", LEGACY_SEEDS_TO_COPY)
print("Trainable seeds:", TRAINABLE_SEEDS)
print("Initial seed:", SEED_TAG)
print("Device:", DEVICE)
print("Text embedding key:", TEXT_EMBED_KEY)
print("Model dim:", MODEL_DIM)
print("Train batch size:", TRAIN_BATCH_SIZE)
print("Eval batch size:", EVAL_BATCH_SIZE)
print("Max epochs:", MAX_EPOCHS)
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)
print("Label smoothing:", LABEL_SMOOTHING)
print("Use AMP:", USE_AMP)
print("Resume training:", RESUME_TRAINING)
print("Clean metadata policy: no question-visible metadata/context IDs are passed to the model.")
print("Adaptive aux gate init probability:", AUX_GATE_INIT_PROB)
print("Visual cache dtype:", VISUAL_CACHE_DTYPE)
print("Reuse saved visual cache:", REUSE_SAVED_VISUAL_CACHE)
print("Force rebuild visual cache:", FORCE_REBUILD_VISUAL_CACHE)
print("Skip train if already done:", SKIP_TRAIN_IF_ALREADY_DONE)
print("Skip final eval if already done:", SKIP_FINAL_EVAL_IF_ALREADY_DONE)
print("Force rerun final eval:", FORCE_RERUN_FINAL_EVAL)
print("Debug rows per split:", DEBUG_MAX_ROWS_PER_SPLIT)


Run seeds: [42, 1337, 2025, 3407]
Legacy seeds to copy: []
Trainable seeds: [42, 1337, 2025, 3407]
Initial seed: seed_42
Device: cuda
Text embedding key: cls_last_hidden_state
Model dim: 512
Train batch size: 96
Eval batch size: 192
Max epochs: 50
Learning rate: 0.0002
Weight decay: 0.0001
Label smoothing: 0.05
Use AMP: True
Resume training: True
Clean metadata policy: no question-visible metadata/context IDs are passed to the model.
Adaptive aux gate init probability: 0.875
Visual cache dtype: torch.float16
Reuse saved visual cache: True
Force rebuild visual cache: False
Skip train if already done: True
Skip final eval if already done: True
Force rerun final eval: False
Debug rows per split: None


# Persistence and Utility Helpers

In [ ]:
def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def now_string():
    return time.strftime("%Y-%m-%d %H:%M:%S")


def read_json(path: Path, default=None):
    if not path.exists():
        return default
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def atomic_write_json(path: Path, payload: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    temp_path = path.with_suffix(path.suffix + ".tmp")
    with open(temp_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, sort_keys=True)
    os.replace(temp_path, path)


def atomic_write_csv(path: Path, rows: list, fieldnames: list):
    path.parent.mkdir(parents=True, exist_ok=True)
    temp_path = path.with_suffix(path.suffix + ".tmp")
    with open(temp_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    os.replace(temp_path, path)


def atomic_torch_save(payload: dict, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    temp_path = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, temp_path)
    os.replace(temp_path, path)


def safe_torch_load(path: Path, map_location="cpu"):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def tensor_bytes(tensor: torch.Tensor):
    return tensor.numel() * tensor.element_size()


def build_seed_paths(run_seed: int):
    run_seed = int(run_seed)
    seed_tag = f"seed_{run_seed}"
    phase4_dir = PHASE4_PARENT_DIR / seed_tag
    checkpoint_dir = phase4_dir / "checkpoints"
    reports_dir = phase4_dir / "reports"
    predictions_dir = phase4_dir / "predictions"
    done_dir = phase4_dir / "done"

    return {
        "SEED": run_seed,
        "SEED_TAG": seed_tag,
        "PHASE4_DIR": phase4_dir,
        "CHECKPOINT_DIR": checkpoint_dir,
        "REPORTS_DIR": reports_dir,
        "PREDICTIONS_DIR": predictions_dir,
        "DONE_DIR": done_dir,
        "PHASE4_CONFIG_PATH": phase4_dir / "phase4_qa_prugtm_hybrid_config.json",
        "INPUT_POLICY_PATH": reports_dir / "input_policy.json",
        "PHASE4_HISTORY_PATH": phase4_dir / "phase4_qa_prugtm_hybrid_training_history.json",
        "PHASE4_REPORT_PATH": phase4_dir / "phase4_qa_prugtm_hybrid_training_report.json",
        "BEST_CKPT_PATH": checkpoint_dir / "best_adaptive_prugtm_qgca_model.pt",
        "LAST_CKPT_PATH": checkpoint_dir / "last_adaptive_prugtm_qgca_model.pt",
        "VAL_METRICS_PATH": reports_dir / "val_metrics.json",
        "TEST_METRICS_PATH": reports_dir / "test_metrics.json",
        "VAL_ATTENTION_PATH": reports_dir / "val_attention_summary.json",
        "TEST_ATTENTION_PATH": reports_dir / "test_attention_summary.json",
        "VAL_QUERY_CONTEXT_PATH": reports_dir / "val_query_context_summary.json",
        "TEST_QUERY_CONTEXT_PATH": reports_dir / "test_query_context_summary.json",
        "VAL_POST_REASONING_PATH": reports_dir / "val_post_reasoning_summary.json",
        "TEST_POST_REASONING_PATH": reports_dir / "test_post_reasoning_summary.json",
        "FINAL_EVAL_SUMMARY_PATH": reports_dir / "final_eval_summary.json",
        "VAL_PREDICTIONS_PATH": predictions_dir / "val_predictions.csv",
        "TEST_PREDICTIONS_PATH": predictions_dir / "test_predictions.csv",
        "TRAINING_DONE_PATH": done_dir / "phase4_qa_prugtm_hybrid_training_complete.json",
        "FINAL_EVAL_DONE_PATH": done_dir / "phase4_qa_prugtm_hybrid_final_eval_complete.json",
    }


def build_input_policy_report(run_seed: int):
    return {
        "created_at": now_string(),
        "phase": "Phase 4 clean-metadata BTUMQA training input policy",
        "dataset_release_name": DATASET_RELEASE_NAME,
        "model_variant": PHASE4_MODEL_VARIANT,
        "run_seed": int(run_seed),
        "seed_tag": f"seed_{int(run_seed)}",
        "policy_name": "strict_no_explicit_metadata_context",
        "phase4_parent_dir": str(PHASE4_PARENT_DIR),
        "input_policy_path": str(INPUT_POLICY_PATH),
        "model_input_fields": CLEAN_MODEL_INPUT_FIELDS,
        "label_fields": CLEAN_LABEL_FIELDS,
        "alignment_only_fields": ALIGNMENT_ONLY_FIELDS,
        "visual_artifact_pointer_fields": VISUAL_ARTIFACT_POINTER_FIELDS,
        "visual_token_source": CLEAN_VISUAL_TOKEN_SOURCE,
        "visual_derived_tensor_inputs": VISUAL_DERIVED_TENSOR_INPUTS,
        "adaptive_uncertainty_guidance": {
            "enabled": True,
            "gate_name": "question_region_aux_gate",
            "gate_inputs": ["question_projection", "learned_region_embeddings"],
            "gate_target": "aux_encoder(region_aux)",
            "gate_shape": "[batch, num_regions, 1]",
            "gate_init_probability": AUX_GATE_INIT_PROB,
            "fusion": "visual_proj(visual_tokens) + aux_gate * aux_encoder(region_aux) + region_embeddings",
            "paper_framing": "question-adaptive uncertainty guidance under metadata-controlled training",
        },
        "selected_context_fields": SELECTED_CONTEXT_FIELDS,
        "forbidden_model_forward_kwargs": FORBIDDEN_MODEL_FORWARD_KWARGS,
        "excluded_audit_only_fields": EXCLUDED_AUDIT_ONLY_FIELDS,
        "audit_reporting_fields_retained_after_inference": AUDIT_REPORTING_FIELDS,
        "allowed_uncertainty_nuance": (
            "Uncertainty is allowed only through visual-derived region_aux tensors, "
            "with question-adaptive gating learned from question embeddings and learned region embeddings, "
            "not as separately embedded tabular/context metadata."
        ),
        "paper_safe_claim": (
            "Answer-proximal metadata/context labels are excluded from model inputs; "
            "audit metadata is retained only for validation and post-inference slice reporting."
        ),
        "checkpoint_safety_note": CHECKPOINT_SAFETY_NOTE,
    }


def assert_clean_model_forward_signature(model):
    forward_arg_names = set(model.forward.__code__.co_varnames[:model.forward.__code__.co_argcount])
    forbidden_present = sorted(forward_arg_names.intersection(FORBIDDEN_MODEL_FORWARD_KWARGS))
    if forbidden_present:
        raise AssertionError(f"Forbidden metadata kwargs remain in model.forward: {forbidden_present}")
    return {
        "checked_at": now_string(),
        "forward_args": sorted(forward_arg_names - {"self"}),
        "forbidden_present": forbidden_present,
        "status": "pass",
    }


def activate_seed_context(run_seed: int):
    context = build_seed_paths(run_seed)
    globals().update(context)

    for path in [PHASE4_DIR, CHECKPOINT_DIR, REPORTS_DIR, PREDICTIONS_DIR, DONE_DIR, VISUAL_CACHE_DIR]:
        path.mkdir(parents=True, exist_ok=True)

    return context


def training_outputs_complete(paths=None):
    paths = build_seed_paths(SEED) if paths is None else paths
    required_paths = [
        paths["BEST_CKPT_PATH"],
        paths["LAST_CKPT_PATH"],
        paths["PHASE4_HISTORY_PATH"],
    ]
    return all(path.exists() for path in required_paths)


def final_eval_outputs_complete(paths=None):
    paths = build_seed_paths(SEED) if paths is None else paths
    required_paths = [
        paths["BEST_CKPT_PATH"],
        paths["VAL_METRICS_PATH"],
        paths["TEST_METRICS_PATH"],
        paths["VAL_ATTENTION_PATH"],
        paths["TEST_ATTENTION_PATH"],
        paths["VAL_QUERY_CONTEXT_PATH"],
        paths["TEST_QUERY_CONTEXT_PATH"],
        paths["VAL_POST_REASONING_PATH"],
        paths["TEST_POST_REASONING_PATH"],
        paths["VAL_PREDICTIONS_PATH"],
        paths["TEST_PREDICTIONS_PATH"],
        paths["FINAL_EVAL_SUMMARY_PATH"],
    ]
    return all(path.exists() for path in required_paths)


def final_eval_is_fresh(paths=None):
    paths = build_seed_paths(SEED) if paths is None else paths
    if not final_eval_outputs_complete(paths):
        return False
    return paths["FINAL_EVAL_SUMMARY_PATH"].stat().st_mtime >= paths["BEST_CKPT_PATH"].stat().st_mtime


def phase4_report_complete(paths=None):
    paths = build_seed_paths(SEED) if paths is None else paths
    return paths["PHASE4_REPORT_PATH"].exists()


def full_seed_outputs_complete(paths=None):
    paths = build_seed_paths(SEED) if paths is None else paths
    return (
        training_outputs_complete(paths)
        and final_eval_outputs_complete(paths)
        and phase4_report_complete(paths)
    )


def copy_legacy_seed_outputs_to_four_seed_dir():
    PHASE4_PARENT_DIR.mkdir(parents=True, exist_ok=True)
    copy_rows = []

    for legacy_seed in LEGACY_SEEDS_TO_COPY:
        legacy_seed = int(legacy_seed)
        seed_tag = f"seed_{legacy_seed}"
        source_dir = PHASE4_LEGACY_PARENT_DIR / seed_tag
        target_dir = PHASE4_PARENT_DIR / seed_tag

        if target_dir.exists():
            if not target_dir.is_dir():
                raise RuntimeError(f"Four-seeds target exists but is not a directory: {target_dir}")
            copy_action = "already_exists"
        else:
            if not source_dir.exists():
                raise FileNotFoundError(f"Missing source legacy seed folder for {seed_tag}: {source_dir}")
            shutil.copytree(source_dir, target_dir)
            copy_action = "copied"

        paths = build_seed_paths(legacy_seed)
        if not full_seed_outputs_complete(paths):
            raise RuntimeError(
                f"Four-seeds legacy output is incomplete after {copy_action} for {seed_tag}: {target_dir}"
            )

        copy_rows.append(
            {
                "run_seed": legacy_seed,
                "seed_tag": seed_tag,
                "copy_action": copy_action,
                "source_dir": str(source_dir),
                "target_dir": str(target_dir),
                "full_seed_complete": bool(full_seed_outputs_complete(paths)),
            }
        )

    return pd.DataFrame(copy_rows)


legacy_seed_copy_df = copy_legacy_seed_outputs_to_four_seed_dir()
print("Legacy seed copy status for four-seeds Phase 4 run:")
print(legacy_seed_copy_df.to_string(index=False))


activate_seed_context(SEED)
seed_everything(SEED)
print("Utility helpers ready.")
print("Active seed context:", SEED_TAG, PHASE4_DIR)


Legacy seed copy status for four-seeds Phase 4 run:
Empty DataFrame
Columns: []
Index: []
Utility helpers ready.
Active seed context: seed_42 /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_4_qgca_training/biasless_metadata_free_training/adaptive_prugtm_qgca_btumqa_four_seeds/seed_42


# Load Phase 3A CSVs and Phase 3B Text Embeddings


In [ ]:
REQUIRED_CSV_COLUMNS = [
    "qa_id",
    "split",
    "patient_id",
    "slice_id",
    "unique_id",
    "question",
    "answer",
    "question_family",
    "question_style",
    "difficulty_level",
    "ambiguity_flag",
    "region_target_primary",
    "region_target_secondary",
    "signal_gap_bucket",
    "decision_rule_id",
    "label_provenance",
    "candidate_keep_reason",
    "phase2c_file",
    "slice_index_in_ugtm_file",
]

REQUIRED_EMBEDDING_KEYS = [
    "qa_id",
    TEXT_EMBED_KEY,
    "answer_id",
    "question_family_id",
    "question_style_id",
    "difficulty_level_id",
]


def ordered_vocab_values(vocab: dict):
    idx_to_value = {idx: value for value, idx in vocab.items()}
    return [idx_to_value[idx] for idx in range(len(idx_to_value))]


def apply_debug_limit(df: pd.DataFrame, split_name: str):
    if DEBUG_MAX_ROWS_PER_SPLIT is None:
        return df.reset_index(drop=True)

    limited = df.iloc[:DEBUG_MAX_ROWS_PER_SPLIT].copy().reset_index(drop=True)
    print(f"[DEBUG] {split_name}: limited rows from {len(df)} to {len(limited)}")
    return limited


def load_split_csv(path: Path, split_name: str):
    if not path.exists():
        raise FileNotFoundError(f"{split_name} CSV not found: {path}")

    df = pd.read_csv(path, dtype=str).fillna("")

    missing = [col for col in REQUIRED_CSV_COLUMNS if col not in df.columns]
    if missing:
        raise ValueError(f"{split_name} missing required columns: {missing}")

    df["patient_id"] = df["patient_id"].astype(str).str.zfill(5)
    df["slice_id"] = df["slice_id"].astype(str).str.zfill(3)
    df["slice_index_in_ugtm_file"] = df["slice_index_in_ugtm_file"].astype(int)
    df = apply_debug_limit(df, split_name)

    print(f"{split_name}: rows={len(df)}")
    print("  question families:", dict(df["question_family"].value_counts().sort_index()))
    print("  question styles:", dict(df["question_style"].value_counts().sort_index()))
    print("  ambiguity flags:", dict(df["ambiguity_flag"].value_counts().sort_index()))
    print("  signal gap buckets:", dict(df["signal_gap_bucket"].value_counts().sort_index()))
    print("  answers:", sorted(df["answer"].unique().tolist()))
    return df


def load_embedding_payload(path: Path, split_name: str, expected_rows: int):
    if not path.exists():
        raise FileNotFoundError(f"{split_name} embeddings not found: {path}")

    payload = safe_torch_load(path, map_location="cpu")

    for key in REQUIRED_EMBEDDING_KEYS:
        if key not in payload:
            raise ValueError(f"{split_name} embeddings missing key: {key}")

    if DEBUG_MAX_ROWS_PER_SPLIT is None:
        if len(payload["qa_id"]) != expected_rows:
            raise ValueError(
                f"{split_name} embeddings row mismatch: "
                f"expected {expected_rows}, found {len(payload['qa_id'])}"
            )
        use_rows = expected_rows
    else:
        use_rows = min(DEBUG_MAX_ROWS_PER_SPLIT, len(payload["qa_id"]))

    embedding_payload = {
        "qa_id": payload["qa_id"][:use_rows],
        "question_embedding": payload[TEXT_EMBED_KEY][:use_rows].float().contiguous(),
        "answer_id": payload["answer_id"][:use_rows].long().contiguous(),
        "question_family_id": payload["question_family_id"][:use_rows].long().contiguous(),
        "question_style_id": payload["question_style_id"][:use_rows].long().contiguous(),
        "difficulty_level_id": payload["difficulty_level_id"][:use_rows].long().contiguous(),
    }

    print(
        f"{split_name} embeddings: "
        f"rows={len(embedding_payload['qa_id'])}, "
        f"shape={tuple(embedding_payload['question_embedding'].shape)}"
    )
    return embedding_payload


answer_vocab = read_json(ANSWER_VOCAB_PATH)
question_family_vocab = read_json(QUESTION_FAMILY_VOCAB_PATH)
question_style_vocab = read_json(QUESTION_STYLE_VOCAB_PATH)
difficulty_level_vocab = read_json(DIFFICULTY_LEVEL_VOCAB_PATH)
ambiguity_flag_vocab = read_json(AMBIGUITY_FLAG_VOCAB_PATH)
signal_gap_bucket_vocab = read_json(SIGNAL_GAP_BUCKET_VOCAB_PATH)
region_target_vocab = read_json(REGION_TARGET_VOCAB_PATH)

required_vocab_map = {
    "answer": answer_vocab,
    "question_family": question_family_vocab,
    "question_style": question_style_vocab,
    "difficulty_level": difficulty_level_vocab,
    "ambiguity_flag": ambiguity_flag_vocab,
    "signal_gap_bucket": signal_gap_bucket_vocab,
    "region_target": region_target_vocab,
}

for vocab_name, vocab in required_vocab_map.items():
    if vocab is None:
        raise FileNotFoundError(f"Missing {vocab_name} vocab JSON")

idx_to_answer = {idx: answer for answer, idx in answer_vocab.items()}
idx_to_question_family = {idx: family for family, idx in question_family_vocab.items()}
idx_to_question_style = {idx: value for value, idx in question_style_vocab.items()}
idx_to_difficulty_level = {idx: value for value, idx in difficulty_level_vocab.items()}
idx_to_ambiguity_flag = {idx: value for value, idx in ambiguity_flag_vocab.items()}
idx_to_signal_gap_bucket = {idx: value for value, idx in signal_gap_bucket_vocab.items()}
idx_to_region_target = {idx: value for value, idx in region_target_vocab.items()}

if ordered_vocab_values(answer_vocab) != FIXED_ANSWER_ORDER:
    raise ValueError(
        "Answer vocabulary order mismatch.\n"
        f"Expected: {FIXED_ANSWER_ORDER}\n"
        f"Found:    {ordered_vocab_values(answer_vocab)}"
    )

if ordered_vocab_values(question_family_vocab) != FIXED_QUESTION_FAMILY_ORDER:
    raise ValueError(
        "Question family vocabulary order mismatch.\n"
        f"Expected: {FIXED_QUESTION_FAMILY_ORDER}\n"
        f"Found:    {ordered_vocab_values(question_family_vocab)}"
    )

if ordered_vocab_values(question_style_vocab) != FIXED_QUESTION_STYLE_ORDER:
    raise ValueError(
        "Question style vocabulary order mismatch.\n"
        f"Expected: {FIXED_QUESTION_STYLE_ORDER}\n"
        f"Found:    {ordered_vocab_values(question_style_vocab)}"
    )

if ordered_vocab_values(difficulty_level_vocab) != FIXED_DIFFICULTY_LEVEL_ORDER:
    raise ValueError(
        "Difficulty vocabulary order mismatch.\n"
        f"Expected: {FIXED_DIFFICULTY_LEVEL_ORDER}\n"
        f"Found:    {ordered_vocab_values(difficulty_level_vocab)}"
    )

if ordered_vocab_values(ambiguity_flag_vocab) != FIXED_AMBIGUITY_FLAG_ORDER:
    raise ValueError(
        "Ambiguity vocabulary order mismatch.\n"
        f"Expected: {FIXED_AMBIGUITY_FLAG_ORDER}\n"
        f"Found:    {ordered_vocab_values(ambiguity_flag_vocab)}"
    )

if ordered_vocab_values(signal_gap_bucket_vocab) != FIXED_SIGNAL_GAP_BUCKET_ORDER:
    raise ValueError(
        "Signal gap vocabulary order mismatch.\n"
        f"Expected: {FIXED_SIGNAL_GAP_BUCKET_ORDER}\n"
        f"Found:    {ordered_vocab_values(signal_gap_bucket_vocab)}"
    )

if ordered_vocab_values(region_target_vocab) != FIXED_REGION_TARGET_ORDER:
    raise ValueError(
        "Region target vocabulary order mismatch.\n"
        f"Expected: {FIXED_REGION_TARGET_ORDER}\n"
        f"Found:    {ordered_vocab_values(region_target_vocab)}"
    )

df_train = load_split_csv(TRAIN_CSV_PATH, "train")
df_val = load_split_csv(VAL_CSV_PATH, "val")
df_test = load_split_csv(TEST_CSV_PATH, "test")

emb_train = load_embedding_payload(
    EMBEDDINGS_TRAIN_PATH,
    "train",
    EXPECTED_SPLIT_ROWS["train"],
)
emb_val = load_embedding_payload(
    EMBEDDINGS_VAL_PATH,
    "val",
    EXPECTED_SPLIT_ROWS["val"],
)
emb_test = load_embedding_payload(
    EMBEDDINGS_TEST_PATH,
    "test",
    EXPECTED_SPLIT_ROWS["test"],
)

for split_name, df, emb in [
    ("train", df_train, emb_train),
    ("val", df_val, emb_val),
    ("test", df_test, emb_test),
]:
    df_qa_ids = df["qa_id"].astype(str).tolist()
    emb_qa_ids = [str(x) for x in emb["qa_id"]]
    if df_qa_ids != emb_qa_ids:
        raise ValueError(f"{split_name} QA ID order mismatch between CSV and Phase 3B embeddings")

print("\nPhase 3A and Phase 3B inputs loaded successfully.")
print("Answer classes:", len(answer_vocab))
print("Question families:", len(question_family_vocab))
print("Question styles:", len(question_style_vocab))
print("Clean metadata policy: no question-visible metadata/context IDs are passed to the model.")


train: rows=157500
  question families: {'ambiguous_subregion_pair': np.int64(15751), 'confidence_qualified_extent': np.int64(11812), 'confidence_qualified_presence': np.int64(11812), 'dominant_region_under_uncertainty': np.int64(15751), 'highest_risk_region': np.int64(11812), 'lowest_risk_region': np.int64(11812), 'more_reliable_region': np.int64(12600), 'more_uncertain_region': np.int64(12600), 'reliability_gap_bucket': np.int64(12600), 'safe_region_for_reasoning': np.int64(12600), 'uncertainty_consistency_check': np.int64(15750), 'uncertainty_gap_bucket': np.int64(12600)}
  question styles: {'ambiguity_sensitive': np.int64(47252), 'bucketed': np.int64(25200), 'comparative': np.int64(25200), 'confidence_qualified': np.int64(23624), 'ranking': np.int64(36224)}
  ambiguity flags: {'no': np.int64(102746), 'yes': np.int64(54754)}
  signal gap buckets: {'close_gap': np.int64(54754), 'moderate_gap': np.int64(64729), 'wide_gap': np.int64(38017)}
  answers: ['ambiguous', 'close_gap', 'confid

# Resolve Phase 2C Paths and Build Original-Token Visual Slice Cache


In [ ]:
def resolve_phase2c_path(raw_path: str, patient_id: str):
    raw_path = str(raw_path).strip()
    candidate = Path(raw_path)

    if candidate.exists():
        return str(candidate)

    fallback_by_patient = PHASE2C_SAVE_DIR / f"{str(patient_id).zfill(5)}.pt"
    if fallback_by_patient.exists():
        return str(fallback_by_patient)

    if candidate.name:
        fallback_by_name = PHASE2C_SAVE_DIR / candidate.name
        if fallback_by_name.exists():
            return str(fallback_by_name)

    raise FileNotFoundError(
        f"Could not resolve Phase 2C file.\n"
        f"raw_path={raw_path}\n"
        f"patient_id={patient_id}\n"
        f"checked={fallback_by_patient}"
    )


def prepare_phase4_dataframe(df: pd.DataFrame, split_name: str):
    df = df.copy().reset_index(drop=True)
    df["phase2c_file_resolved"] = [
        resolve_phase2c_path(raw_path, patient_id)
        for raw_path, patient_id in zip(df["phase2c_file"], df["patient_id"])
    ]
    print(f"{split_name}: resolved Phase 2C paths for {len(df)} rows")
    return df


df_train = prepare_phase4_dataframe(df_train, "train")
df_val = prepare_phase4_dataframe(df_val, "val")
df_test = prepare_phase4_dataframe(df_test, "test")

combined_visual_df = pd.concat(
    [
        df_train[["unique_id", "patient_id", "phase2c_file_resolved", "slice_index_in_ugtm_file"]],
        df_val[["unique_id", "patient_id", "phase2c_file_resolved", "slice_index_in_ugtm_file"]],
        df_test[["unique_id", "patient_id", "phase2c_file_resolved", "slice_index_in_ugtm_file"]],
    ],
    axis=0,
    ignore_index=True,
)

unique_visual_df = combined_visual_df.drop_duplicates(
    subset=["unique_id", "phase2c_file_resolved", "slice_index_in_ugtm_file"]
).reset_index(drop=True)

consistency_check = unique_visual_df.groupby("unique_id").agg(
    num_phase2c_files=("phase2c_file_resolved", "nunique"),
    num_slice_indices=("slice_index_in_ugtm_file", "nunique"),
).reset_index()

bad_consistency = consistency_check[
    (consistency_check["num_phase2c_files"] != 1) |
    (consistency_check["num_slice_indices"] != 1)
]

if len(bad_consistency) > 0:
    raise ValueError(
        f"Found inconsistent unique_id linkage in Phase 4 inputs: {len(bad_consistency)} rows"
    )

expected_unique_ids = unique_visual_df["unique_id"].astype(str).tolist()
expected_unique_id_set = set(expected_unique_ids)

print("Unique visual slices needed:", len(expected_unique_ids))
print("Unique patient files to read if rebuild is needed:", unique_visual_df["phase2c_file_resolved"].nunique())


def validate_visual_cache_payload(payload: dict, expected_unique_id_set: set):
    required_keys = ["unique_ids", "original_tokens", "region_aux", "region_names", "created_at"]
    for key in required_keys:
        if key not in payload:
            return False, f"missing_key_{key}"

    unique_ids = [str(x) for x in payload["unique_ids"]]
    unique_id_set = set(unique_ids)

    missing_ids = sorted(expected_unique_id_set.difference(unique_id_set))
    if missing_ids:
        return False, f"missing_expected_unique_ids_count_{len(missing_ids)}"

    original_tokens = payload["original_tokens"]
    region_aux = payload["region_aux"]

    if len(original_tokens.shape) != 3:
        return False, f"bad_original_tokens_shape_{list(original_tokens.shape)}"

    if len(region_aux.shape) != 3:
        return False, f"bad_region_aux_shape_{list(region_aux.shape)}"

    if original_tokens.shape[0] != len(unique_ids):
        return False, "original_tokens_row_mismatch"

    if region_aux.shape[0] != len(unique_ids):
        return False, "region_aux_row_mismatch"

    if original_tokens.shape[1:] != (NUM_VISUAL_TOKENS, VISUAL_TOKEN_DIM):
        return False, f"bad_original_token_tail_shape_{list(original_tokens.shape[1:])}"

    if region_aux.shape[1:] != (NUM_VISUAL_TOKENS, AUX_FEATURE_DIM):
        return False, f"bad_region_aux_tail_shape_{list(region_aux.shape[1:])}"

    if torch.isnan(original_tokens.float()).any() or torch.isinf(original_tokens.float()).any():
        return False, "nan_or_inf_original_tokens"

    if torch.isnan(region_aux.float()).any() or torch.isinf(region_aux.float()).any():
        return False, "nan_or_inf_region_aux"

    if payload["region_names"] != REGION_NAMES:
        return False, "region_name_order_mismatch"

    return True, "ok"


def save_visual_cache_payload(payload: dict):
    atomic_torch_save(payload, VISUAL_CACHE_TENSOR_PATH)
    metadata = {
        "created_at": payload["created_at"],
        "dataset_release_name": DATASET_RELEASE_NAME,
        "cached_unique_slices": int(len(payload["unique_ids"])),
        "cache_dtype": str(payload["original_tokens"].dtype),
        "original_tokens_shape": list(payload["original_tokens"].shape),
        "region_aux_shape": list(payload["region_aux"].shape),
        "region_names": payload["region_names"],
        "aux_features": payload["aux_features"],
        "token_field": "original_tokens",
        "source": payload.get("source", "rebuilt"),
    }
    atomic_write_json(VISUAL_CACHE_METADATA_PATH, metadata)


def build_visual_cache_payload(unique_rows_df: pd.DataFrame):
    unique_id_rows = []
    original_token_rows = []
    region_aux_rows = []

    grouped = unique_rows_df.groupby("phase2c_file_resolved", sort=True)
    total_groups = unique_rows_df["phase2c_file_resolved"].nunique()

    for phase2c_path, group_df in tqdm(grouped, total=total_groups, desc="Building original-token visual cache"):
        payload = safe_torch_load(Path(phase2c_path), map_location="cpu")

        required_keys = [
            "unique_ids",
            "original_tokens",
            "region_uncertainty",
            "modulation_weights",
            "region_present",
            "region_area_pixels",
        ]
        for key in required_keys:
            if key not in payload:
                raise ValueError(f"Missing key in Phase 2C payload: {key} | file={phase2c_path}")

        unique_ids_in_file = payload["unique_ids"]
        original_tokens = payload["original_tokens"]
        region_uncertainty = payload["region_uncertainty"]
        modulation_weights = payload["modulation_weights"]
        region_present = payload["region_present"]
        region_area_pixels = payload["region_area_pixels"]

        num_slices = len(unique_ids_in_file)

        for row in group_df.itertuples(index=False):
            unique_id = str(row.unique_id)
            slice_index = int(row.slice_index_in_ugtm_file)

            if slice_index < 0 or slice_index >= num_slices:
                raise IndexError(
                    f"Slice index out of bounds for {unique_id}: "
                    f"slice_index={slice_index}, num_slices={num_slices}, file={phase2c_path}"
                )

            if STRICT_PHASE2C_ALIGNMENT:
                file_unique_id = str(unique_ids_in_file[slice_index])
                if file_unique_id != unique_id:
                    raise ValueError(
                        f"Phase 2C alignment mismatch: expected {unique_id}, found {file_unique_id}, "
                        f"file={phase2c_path}, index={slice_index}"
                    )

            vt = original_tokens[slice_index].detach().cpu().to(VISUAL_CACHE_DTYPE).contiguous()

            uncertainty = region_uncertainty[slice_index].detach().cpu().float().contiguous()
            weights = modulation_weights[slice_index].detach().cpu().float().contiguous()
            present = region_present[slice_index].detach().cpu().float().contiguous()
            area_pixels = region_area_pixels[slice_index].detach().cpu().float().contiguous()

            if vt.shape != (NUM_VISUAL_TOKENS, VISUAL_TOKEN_DIM):
                raise ValueError(f"Bad original token shape for {unique_id}: {tuple(vt.shape)}")

            global_area = max(float(area_pixels[-1].item()), 1.0)
            relative_area = torch.clamp(area_pixels / global_area, min=0.0, max=1.0)

            aux = torch.stack(
                [uncertainty, weights, present, relative_area],
                dim=-1,
            ).to(VISUAL_CACHE_DTYPE).contiguous()

            unique_id_rows.append(unique_id)
            original_token_rows.append(vt)
            region_aux_rows.append(aux)

        del payload
        gc.collect()

    original_tokens_tensor = torch.stack(original_token_rows, dim=0).contiguous()
    region_aux_tensor = torch.stack(region_aux_rows, dim=0).contiguous()

    payload = {
        "created_at": now_string(),
        "dataset_release_name": DATASET_RELEASE_NAME,
        "unique_ids": unique_id_rows,
        "original_tokens": original_tokens_tensor,
        "region_aux": region_aux_tensor,
        "region_names": REGION_NAMES,
        "aux_features": ["uncertainty", "modulation_weight", "region_present", "relative_area"],
        "token_field": "original_tokens",
        "source": "rebuilt_from_phase2c",
    }
    return payload


def payload_to_visual_cache(payload: dict, required_unique_ids: list):
    payload_unique_ids = [str(x) for x in payload["unique_ids"]]
    payload_index = {uid: idx for idx, uid in enumerate(payload_unique_ids)}

    visual_cache = {}
    approx_bytes = 0

    for uid in required_unique_ids:
        idx = payload_index[uid]
        vt = payload["original_tokens"][idx]
        aux = payload["region_aux"][idx]

        visual_cache[uid] = {
            "visual_tokens": vt,
            "region_aux": aux,
        }
        approx_bytes += tensor_bytes(vt) + tensor_bytes(aux)

    summary = {
        "created_at": now_string(),
        "dataset_release_name": DATASET_RELEASE_NAME,
        "cached_unique_slices": int(len(visual_cache)),
        "source_rows_in_saved_payload": int(len(payload_unique_ids)),
        "approx_materialized_memory_mb": round(approx_bytes / (1024 ** 2), 2),
        "cache_dtype": str(payload["original_tokens"].dtype),
        "region_names": REGION_NAMES,
        "aux_features": ["uncertainty", "modulation_weight", "region_present", "relative_area"],
        "token_field": "original_tokens",
        "source": payload.get("source", "loaded_from_disk"),
    }
    return visual_cache, summary


visual_cache_payload = None
loaded_from_disk = False

if REUSE_SAVED_VISUAL_CACHE and VISUAL_CACHE_TENSOR_PATH.exists() and not FORCE_REBUILD_VISUAL_CACHE:
    print("Found existing original-token visual cache payload. Validating:", VISUAL_CACHE_TENSOR_PATH)
    candidate_payload = safe_torch_load(VISUAL_CACHE_TENSOR_PATH, map_location="cpu")
    is_valid, reason = validate_visual_cache_payload(candidate_payload, expected_unique_id_set)

    if is_valid:
        visual_cache_payload = candidate_payload
        loaded_from_disk = True
        print("Original-token visual cache payload is valid and will be reused.")
    else:
        print(f"Existing original-token visual cache payload is invalid for current run: {reason}")
        print("Will rebuild the visual cache payload from Phase 2C files.")

if visual_cache_payload is None:
    visual_cache_payload = build_visual_cache_payload(unique_visual_df)
    save_visual_cache_payload(visual_cache_payload)
    print("Saved rebuilt original-token visual cache payload:", VISUAL_CACHE_TENSOR_PATH)

visual_slice_cache, visual_cache_summary = payload_to_visual_cache(
    payload=visual_cache_payload,
    required_unique_ids=expected_unique_ids,
)

visual_cache_summary["loaded_from_disk"] = bool(loaded_from_disk)
visual_cache_summary["visual_cache_tensor_path"] = str(VISUAL_CACHE_TENSOR_PATH)
visual_cache_summary["visual_cache_metadata_path"] = str(VISUAL_CACHE_METADATA_PATH)

atomic_write_json(VISUAL_CACHE_METADATA_PATH, visual_cache_summary)

expected_all_unique_ids = set(
    pd.concat([df_train["unique_id"], df_val["unique_id"], df_test["unique_id"]]).astype(str).tolist()
)
missing_in_cache = sorted(expected_all_unique_ids.difference(set(visual_slice_cache.keys())))
if missing_in_cache:
    raise ValueError(f"Missing cached visual slices: {missing_in_cache[:10]}")

print(json.dumps(visual_cache_summary, indent=2))
print("Original-token visual slice cache ready.")


train: resolved Phase 2C paths for 157500 rows
val: resolved Phase 2C paths for 33750 rows
test: resolved Phase 2C paths for 33750 rows
Unique visual slices needed: 72025
Unique patient files to read if rebuild is needed: 1251
Found existing original-token visual cache payload. Validating: /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_4_qgca_training/single_stage_btumqa_225k_prugtm_45e/visual_cache/visual_slice_cache_original_tokens.pt
Original-token visual cache payload is valid and will be reused.
{
  "created_at": "2026-05-12 21:14:07",
  "dataset_release_name": "BTUMQA-225K",
  "cached_unique_slices": 72025,
  "source_rows_in_saved_payload": 72025,
  "approx_materialized_memory_mb": 425.32,
  "cache_dtype": "torch.float16",
  "region_names": [
    "edema",
    "ncr_net",
    "enhancing",
    "tumor",
    "context",
    "global"
  ],
  "aux_features": [
    "uncertainty",
    "modulation_weight",
    "region_present",
   

# Build Phase 4 Datasets and DataLoaders


In [ ]:
def make_region_target_pair(primary: str, secondary: str):
    primary_value = str(primary)
    secondary_value = str(secondary)
    if secondary_value == "":
        secondary_value = "none"
    return f"{primary_value}|{secondary_value}"


class Phase4QGCADataset(Dataset):
    def __init__(self, df: pd.DataFrame, embedding_payload: dict, visual_cache: dict):
        self.df = df.reset_index(drop=True)
        self.visual_cache = visual_cache

        self.qa_ids = [str(x) for x in embedding_payload["qa_id"]]
        self.question_embeddings = embedding_payload["question_embedding"].contiguous()
        self.answer_ids = embedding_payload["answer_id"].contiguous()
        self.question_family_ids = embedding_payload["question_family_id"].contiguous()
        self.question_style_ids = embedding_payload["question_style_id"].contiguous()
        self.difficulty_level_ids = embedding_payload["difficulty_level_id"].contiguous()

        self.unique_ids = self.df["unique_id"].astype(str).tolist()
        self.answer_texts = self.df["answer"].astype(str).tolist()
        self.question_family_texts = self.df["question_family"].astype(str).tolist()
        self.question_style_texts = self.df["question_style"].astype(str).tolist()
        self.difficulty_level_texts = self.df["difficulty_level"].astype(str).tolist()
        self.ambiguity_flag_texts = self.df["ambiguity_flag"].astype(str).tolist()
        self.signal_gap_bucket_texts = self.df["signal_gap_bucket"].astype(str).tolist()
        self.region_target_primary_texts = self.df["region_target_primary"].astype(str).tolist()
        self.region_target_secondary_texts = self.df["region_target_secondary"].astype(str).tolist()
        self.region_target_pair_texts = [
            make_region_target_pair(primary, secondary)
            for primary, secondary in zip(self.region_target_primary_texts, self.region_target_secondary_texts)
        ]

        if len(self.df) != len(self.qa_ids):
            raise ValueError("Dataset length mismatch between CSV rows and embedding rows")

    def __len__(self):
        return len(self.qa_ids)

    def __getitem__(self, index: int):
        unique_id = self.unique_ids[index]
        cached = self.visual_cache[unique_id]

        return {
            "qa_id": self.qa_ids[index],
            "unique_id": unique_id,
            "question_embedding": self.question_embeddings[index],
            "visual_tokens": cached["visual_tokens"],
            "region_aux": cached["region_aux"],
            "answer_id": self.answer_ids[index],
            "question_family_id": self.question_family_ids[index],
            "question_style_id": self.question_style_ids[index],
            "difficulty_level_id": self.difficulty_level_ids[index],
            "answer_text": self.answer_texts[index],
            "question_family_text": self.question_family_texts[index],
            "question_style_text": self.question_style_texts[index],
            "difficulty_level_text": self.difficulty_level_texts[index],
            "ambiguity_flag_text": self.ambiguity_flag_texts[index],
            "signal_gap_bucket_text": self.signal_gap_bucket_texts[index],
            "region_target_primary_text": self.region_target_primary_texts[index],
            "region_target_secondary_text": self.region_target_secondary_texts[index],
            "region_target_pair_text": self.region_target_pair_texts[index],
        }


def build_class_weights(answer_ids: torch.Tensor, num_classes: int):
    counts = torch.bincount(answer_ids, minlength=num_classes).float()
    weights = counts.sum() / counts.clamp_min(1.0)
    weights = weights / weights.mean()
    return weights, counts


train_dataset = Phase4QGCADataset(df_train, emb_train, visual_slice_cache)
val_dataset = Phase4QGCADataset(df_val, emb_val, visual_slice_cache)
test_dataset = Phase4QGCADataset(df_test, emb_test, visual_slice_cache)

train_loader = DataLoader(
    train_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

class_weights, class_counts = build_class_weights(train_dataset.answer_ids, len(answer_vocab))

print("Train rows:", len(train_dataset))
print("Val rows:", len(val_dataset))
print("Test rows:", len(test_dataset))
print("Train steps per epoch:", math.ceil(len(train_dataset) / TRAIN_BATCH_SIZE))
print("Val steps:", math.ceil(len(val_dataset) / EVAL_BATCH_SIZE))
print("Test steps:", math.ceil(len(test_dataset) / EVAL_BATCH_SIZE))
print()

print("Train class distribution:")
for class_idx in range(len(answer_vocab)):
    answer_name = idx_to_answer[class_idx]
    count = int(class_counts[class_idx].item())
    weight = float(class_weights[class_idx].item())
    print(f"  {class_idx:02d} | {answer_name:24s} | count={count:6d} | weight={weight:.4f}")

sample_batch = next(iter(train_loader))
print("\nSample batch shapes:")
print("  question_embedding:", tuple(sample_batch["question_embedding"].shape))
print("  visual_tokens:", tuple(sample_batch["visual_tokens"].shape))
print("  region_aux:", tuple(sample_batch["region_aux"].shape))
print("  answer_id:", tuple(sample_batch["answer_id"].shape))
del sample_batch
gc.collect()


Train rows: 157500
Val rows: 33750
Test rows: 33750
Train steps per epoch: 1641
Val steps: 176
Test steps: 176

Train class distribution:
  00 | ambiguous                | count=  3235 | weight=0.9282
  01 | close_gap                | count= 16310 | weight=0.1841
  02 | confident_present        | count=  3939 | weight=0.7623
  03 | consistent               | count= 14418 | weight=0.2083
  04 | context                  | count= 14899 | weight=0.2015
  05 | distinct                 | count= 12516 | weight=0.2399
  06 | edema                    | count= 17013 | weight=0.1765
  07 | enhancing                | count= 16077 | weight=0.1868
  08 | global                   | count=  2294 | weight=1.3090
  09 | large_confident          | count=  1688 | weight=1.7790
  10 | large_uncertain          | count=  1688 | weight=1.7790
  11 | moderate_confident       | count=  1688 | weight=1.7790
  12 | moderate_gap             | count=  7121 | weight=0.4217
  13 | moderate_uncertain       | count=  1

52

# Define the Phase 4 Single-Stage BTUMQA Hybrid Post-Reasoning QGCA Model


In [ ]:
class Phase4SingleStageBTUMQAQAPRUGTMHybridModel(nn.Module):
    def __init__(
        self,
        text_embed_dim: int,
        visual_token_dim: int,
        model_dim: int,
        aux_feature_dim: int,
        num_regions: int,
        num_attention_heads: int,
        classifier_hidden_dim: int,
        aux_hidden_dim: int,
        dropout: float,
        num_answers: int,
        num_question_styles: int,
        context_embed_dim: int,
        context_hidden_dim: int,
    ):
        super().__init__()

        self.question_proj = nn.Sequential(
            nn.Linear(text_embed_dim, model_dim),
            nn.LayerNorm(model_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.visual_proj = nn.Sequential(
            nn.Linear(visual_token_dim, model_dim),
            nn.LayerNorm(model_dim),
        )
        self.aux_encoder = nn.Sequential(
            nn.Linear(aux_feature_dim, aux_hidden_dim),
            nn.GELU(),
            nn.Linear(aux_hidden_dim, model_dim),
        )
        self.question_region_aux_gate = nn.Sequential(
            nn.Linear(model_dim * 2, model_dim),
            nn.LayerNorm(model_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(model_dim, 1),
        )
        self._initialize_aux_gate(AUX_GATE_INIT_PROB)

        self.clean_metadata_policy = {
            "model_inputs": CLEAN_MODEL_INPUT_FIELDS,
            "excluded_context_fields": EXCLUDED_AUDIT_ONLY_FIELDS,
            "audit_reporting_fields": AUDIT_REPORTING_FIELDS,
            "adaptive_uncertainty_gate": {
                "inputs": ["question_projection", "learned_region_embeddings"],
                "target": "aux_encoder(region_aux)",
                "shape": "[batch, num_regions, 1]",
                "init_probability": AUX_GATE_INIT_PROB,
            },
        }

        self.region_embeddings = nn.Parameter(torch.randn(num_regions, model_dim) * 0.02)
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=model_dim,
            num_heads=num_attention_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.attention_fusion_norm = nn.LayerNorm(model_dim)

        self.classifier = nn.Sequential(
            nn.Linear(model_dim * 6, classifier_hidden_dim),
            nn.LayerNorm(classifier_hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(classifier_hidden_dim, num_answers),
        )

    def _initialize_aux_gate(self, init_probability: float):
        init_probability = float(np.clip(init_probability, 1e-4, 1.0 - 1e-4))
        init_logit = float(np.log(init_probability / (1.0 - init_probability)))
        final_layer = self.question_region_aux_gate[-1]
        nn.init.normal_(final_layer.weight, mean=0.0, std=1e-3)
        nn.init.constant_(final_layer.bias, init_logit)

    def forward(
        self,
        question_embedding,
        visual_tokens,
        region_aux,
    ):
        q = self.question_proj(question_embedding)
        visual_base = self.visual_proj(visual_tokens)
        aux_features = self.aux_encoder(region_aux)

        region_embeddings = self.region_embeddings.unsqueeze(0).expand(visual_tokens.shape[0], -1, -1)
        question_by_region = q.unsqueeze(1).expand(-1, visual_tokens.shape[1], -1)
        aux_gate_inputs = torch.cat([question_by_region, region_embeddings], dim=-1)
        aux_gate = torch.sigmoid(self.question_region_aux_gate(aux_gate_inputs))

        adaptive_aux_features = aux_gate * aux_features
        v = visual_base + adaptive_aux_features + region_embeddings

        uncertainty_context = adaptive_aux_features.mean(dim=1)
        query_gate = aux_gate.squeeze(-1).mean(dim=-1)
        conditioned_query = q

        attn_output, attn_weights = self.cross_attention(
            query=conditioned_query.unsqueeze(1),
            key=v,
            value=v,
            need_weights=True,
            average_attn_weights=True,
        )

        attention_weights = attn_weights.squeeze(1)
        attended_visual = self.attention_fusion_norm(attn_output.squeeze(1) + conditioned_query)

        region_uncertainty = torch.clamp(region_aux[..., 0], min=0.0, max=1.0)
        attended_uncertainty = torch.sum(attention_weights * region_uncertainty, dim=-1)
        attended_uncertainty = torch.clamp(attended_uncertainty, min=0.0, max=1.0)
        post_reasoning_gate = 1.0 - attended_uncertainty
        gated_visual = attended_visual * post_reasoning_gate.unsqueeze(-1)

        classifier_input = torch.cat(
            [
                gated_visual,
                q,
                conditioned_query,
                gated_visual * conditioned_query,
                torch.abs(gated_visual - conditioned_query),
                uncertainty_context,
            ],
            dim=-1,
        )

        logits = self.classifier(classifier_input)

        return {
            "logits": logits,
            "attention_weights": attention_weights,
            "refined_features": gated_visual,
            "question_features": q,
            "conditioned_query": conditioned_query,
            "uncertainty_context": uncertainty_context,
            "attended_visual": attended_visual,
            "attended_uncertainty": attended_uncertainty,
            "post_reasoning_gate": post_reasoning_gate,
            "aux_gate": aux_gate,
            "aux_gate_mean": query_gate,
            "query_gate_mean": query_gate,
        }


print("Phase 4 clean-metadata BTUMQA QA-PRUGTM hybrid adaptive uncertainty QGCA model class defined.")

Phase 4 clean-metadata BTUMQA QA-PRUGTM hybrid adaptive uncertainty QGCA model class defined.


# Define Training, Validation, and Post-Reasoning Reporting Helpers


In [ ]:
def normalize_group_label(value):
    value = str(value)
    return value if value != "" else "<empty>"


def move_batch_to_device(batch: dict):
    return {
        "question_embedding": batch["question_embedding"].to(DEVICE, non_blocking=True).float(),
        "visual_tokens": batch["visual_tokens"].to(DEVICE, non_blocking=True).float(),
        "region_aux": batch["region_aux"].to(DEVICE, non_blocking=True).float(),
        "answer_id": batch["answer_id"].to(DEVICE, non_blocking=True).long(),
        "qa_id": batch["qa_id"],
        "unique_id": batch["unique_id"],
        "answer_text": batch["answer_text"],
        "question_family_text": batch["question_family_text"],
        "question_style_text": batch["question_style_text"],
        "difficulty_level_text": batch["difficulty_level_text"],
        "ambiguity_flag_text": batch["ambiguity_flag_text"],
        "signal_gap_bucket_text": batch["signal_gap_bucket_text"],
        "region_target_primary_text": batch["region_target_primary_text"],
        "region_target_secondary_text": batch["region_target_secondary_text"],
        "region_target_pair_text": batch["region_target_pair_text"],
    }


def compute_group_metrics(gold_ids, pred_ids, group_labels):
    gold_ids = np.asarray(gold_ids, dtype=np.int64)
    pred_ids = np.asarray(pred_ids, dtype=np.int64)
    group_labels = np.asarray([normalize_group_label(x) for x in group_labels], dtype=object)

    grouped = {}
    for group_label in sorted(np.unique(group_labels).tolist()):
        mask = group_labels == group_label
        grouped[group_label] = {
            "count": int(mask.sum()),
            "accuracy": float(accuracy_score(gold_ids[mask], pred_ids[mask])),
            "macro_f1": float(f1_score(gold_ids[mask], pred_ids[mask], average="macro", zero_division=0)),
        }
    return grouped


def compute_metrics(
    gold_ids,
    pred_ids,
    family_labels,
    question_style_labels,
    difficulty_labels,
    ambiguity_labels,
    signal_gap_labels,
    region_pair_labels,
):
    gold_ids = np.asarray(gold_ids, dtype=np.int64)
    pred_ids = np.asarray(pred_ids, dtype=np.int64)

    metrics = {
        "accuracy": float(accuracy_score(gold_ids, pred_ids)),
        "macro_f1": float(f1_score(gold_ids, pred_ids, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(gold_ids, pred_ids, average="weighted", zero_division=0)),
    }

    metrics["per_question_family"] = compute_group_metrics(gold_ids, pred_ids, family_labels)
    metrics["per_question_style"] = compute_group_metrics(gold_ids, pred_ids, question_style_labels)
    metrics["per_difficulty_level"] = compute_group_metrics(gold_ids, pred_ids, difficulty_labels)
    metrics["per_ambiguity_flag"] = compute_group_metrics(gold_ids, pred_ids, ambiguity_labels)
    metrics["per_signal_gap_bucket"] = compute_group_metrics(gold_ids, pred_ids, signal_gap_labels)
    metrics["per_region_target_pair"] = compute_group_metrics(gold_ids, pred_ids, region_pair_labels)
    return metrics


def summarize_attention(attention_rows, family_labels, ambiguity_labels, signal_gap_labels):
    attention_tensor = torch.tensor(attention_rows, dtype=torch.float32)
    family_labels = [normalize_group_label(x) for x in family_labels]
    ambiguity_labels = [normalize_group_label(x) for x in ambiguity_labels]
    signal_gap_labels = [normalize_group_label(x) for x in signal_gap_labels]

    overall_mean = attention_tensor.mean(dim=0)

    def by_group(labels):
        grouped = {}
        for label in sorted(set(labels)):
            mask = torch.tensor([item == label for item in labels], dtype=torch.bool)
            label_mean = attention_tensor[mask].mean(dim=0)
            grouped[label] = {
                region_name: round(float(label_mean[idx].item()), 6)
                for idx, region_name in enumerate(REGION_NAMES)
            }
        return grouped

    return {
        "overall_mean_attention": {
            region_name: round(float(overall_mean[idx].item()), 6)
            for idx, region_name in enumerate(REGION_NAMES)
        },
        "by_question_family": by_group(family_labels),
        "by_ambiguity_flag": by_group(ambiguity_labels),
        "by_signal_gap_bucket": by_group(signal_gap_labels),
    }


def summarize_query_context(query_gate_rows, family_labels, ambiguity_labels, signal_gap_labels):
    if len(query_gate_rows) == 0:
        return {}

    query_gate_rows = np.asarray(query_gate_rows, dtype=np.float32)
    family_labels = [normalize_group_label(x) for x in family_labels]
    ambiguity_labels = [normalize_group_label(x) for x in ambiguity_labels]
    signal_gap_labels = [normalize_group_label(x) for x in signal_gap_labels]

    def by_group(labels):
        grouped = {}
        label_array = np.asarray(labels, dtype=object)
        for label in sorted(set(labels)):
            mask = label_array == label
            grouped[label] = {
                "count": int(mask.sum()),
                "mean_aux_gate": round(float(query_gate_rows[mask].mean()), 6),
            }
        return grouped

    return {
        "summary_name": "question_region_adaptive_aux_gate",
        "overall_mean_aux_gate": round(float(query_gate_rows.mean()), 6),
        "by_question_family": by_group(family_labels),
        "by_ambiguity_flag": by_group(ambiguity_labels),
        "by_signal_gap_bucket": by_group(signal_gap_labels),
        "gate_shape": "[batch, num_regions, 1]",
        "gate_value_range": "sigmoid bounded [0, 1]",
    }

def summarize_post_reasoning(
    attended_uncertainty_rows,
    post_reasoning_gate_rows,
    family_labels,
    ambiguity_labels,
    signal_gap_labels,
    region_pair_labels,
):
    if len(attended_uncertainty_rows) == 0:
        return {}

    attended_uncertainty_rows = np.asarray(attended_uncertainty_rows, dtype=np.float32)
    post_reasoning_gate_rows = np.asarray(post_reasoning_gate_rows, dtype=np.float32)
    family_labels = [normalize_group_label(x) for x in family_labels]
    ambiguity_labels = [normalize_group_label(x) for x in ambiguity_labels]
    signal_gap_labels = [normalize_group_label(x) for x in signal_gap_labels]
    region_pair_labels = [normalize_group_label(x) for x in region_pair_labels]

    def by_group(labels):
        grouped = {}
        label_array = np.asarray(labels, dtype=object)
        for label in sorted(set(labels)):
            mask = label_array == label
            grouped[label] = {
                "count": int(mask.sum()),
                "mean_attended_uncertainty": round(float(attended_uncertainty_rows[mask].mean()), 6),
                "mean_post_reasoning_gate": round(float(post_reasoning_gate_rows[mask].mean()), 6),
            }
        return grouped

    return {
        "overall_mean_attended_uncertainty": round(float(attended_uncertainty_rows.mean()), 6),
        "overall_mean_post_reasoning_gate": round(float(post_reasoning_gate_rows.mean()), 6),
        "by_question_family": by_group(family_labels),
        "by_ambiguity_flag": by_group(ambiguity_labels),
        "by_signal_gap_bucket": by_group(signal_gap_labels),
        "by_region_target_pair": by_group(region_pair_labels),
    }


def train_one_epoch(model, loader, optimizer, criterion, scaler, epoch_index):
    model.train()

    running_loss = 0.0
    total_rows = 0

    progress = tqdm(loader, desc=f"Train Epoch {epoch_index + 1}", leave=False)

    for batch in progress:
        batch = move_batch_to_device(batch)
        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type="cuda" if torch.cuda.is_available() else "cpu", enabled=USE_AMP):
            outputs = model(
                question_embedding=batch["question_embedding"],
                visual_tokens=batch["visual_tokens"],
                region_aux=batch["region_aux"],
            )
            logits = outputs["logits"]
            loss = criterion(logits, batch["answer_id"])

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        scaler.step(optimizer)
        scaler.update()

        batch_size = batch["answer_id"].shape[0]
        running_loss += float(loss.item()) * batch_size
        total_rows += batch_size

        progress.set_postfix({"loss": round(running_loss / max(total_rows, 1), 5)})

    return {
        "loss": round(running_loss / max(total_rows, 1), 6),
    }


@torch.no_grad()
def evaluate_model(model, loader, criterion, split_name, return_predictions=False):
    model.eval()

    running_loss = 0.0
    total_rows = 0

    gold_ids = []
    pred_ids = []
    family_labels = []
    question_style_labels = []
    difficulty_labels = []
    ambiguity_labels = []
    signal_gap_labels = []
    region_pair_labels = []
    attention_rows = []
    query_gate_rows = []
    attended_uncertainty_rows = []
    post_reasoning_gate_rows = []
    prediction_rows = []

    progress = tqdm(loader, desc=f"Evaluate {split_name}", leave=False)

    for batch in progress:
        moved = move_batch_to_device(batch)

        outputs = model(
            question_embedding=moved["question_embedding"],
            visual_tokens=moved["visual_tokens"],
            region_aux=moved["region_aux"],
        )

        logits = outputs["logits"]
        loss = criterion(logits, moved["answer_id"])

        probs = torch.softmax(logits, dim=-1)
        conf, preds = probs.max(dim=-1)
        attention = outputs["attention_weights"]
        query_gate_mean = outputs["query_gate_mean"]
        attended_uncertainty = outputs["attended_uncertainty"]
        post_reasoning_gate = outputs["post_reasoning_gate"]

        batch_size = moved["answer_id"].shape[0]
        running_loss += float(loss.item()) * batch_size
        total_rows += batch_size

        gold_cpu = moved["answer_id"].detach().cpu().numpy().tolist()
        pred_cpu = preds.detach().cpu().numpy().tolist()
        conf_cpu = conf.detach().cpu().numpy().tolist()
        attention_cpu = attention.detach().cpu().numpy().tolist()
        query_gate_cpu = query_gate_mean.detach().cpu().numpy().tolist()
        attended_uncertainty_cpu = attended_uncertainty.detach().cpu().numpy().tolist()
        post_reasoning_gate_cpu = post_reasoning_gate.detach().cpu().numpy().tolist()

        gold_ids.extend(gold_cpu)
        pred_ids.extend(pred_cpu)
        family_labels.extend(moved["question_family_text"])
        question_style_labels.extend(moved["question_style_text"])
        difficulty_labels.extend(moved["difficulty_level_text"])
        ambiguity_labels.extend(moved["ambiguity_flag_text"])
        signal_gap_labels.extend(moved["signal_gap_bucket_text"])
        region_pair_labels.extend(moved["region_target_pair_text"])
        attention_rows.extend(attention_cpu)
        query_gate_rows.extend(query_gate_cpu)
        attended_uncertainty_rows.extend(attended_uncertainty_cpu)
        post_reasoning_gate_rows.extend(post_reasoning_gate_cpu)

        if return_predictions:
            for idx in range(batch_size):
                prediction_rows.append({
                    "split": split_name,
                    "qa_id": moved["qa_id"][idx],
                    "unique_id": moved["unique_id"][idx],
                    "question_family": moved["question_family_text"][idx],
                    "question_style": moved["question_style_text"][idx],
                    "difficulty_level": moved["difficulty_level_text"][idx],
                    "ambiguity_flag": moved["ambiguity_flag_text"][idx],
                    "signal_gap_bucket": moved["signal_gap_bucket_text"][idx],
                    "region_target_primary": moved["region_target_primary_text"][idx],
                    "region_target_secondary": moved["region_target_secondary_text"][idx],
                    "region_target_pair": moved["region_target_pair_text"][idx],
                    "gold_answer": idx_to_answer[gold_cpu[idx]],
                    "predicted_answer": idx_to_answer[pred_cpu[idx]],
                    "confidence": round(float(conf_cpu[idx]), 6),
                    "attended_uncertainty": round(float(attended_uncertainty_cpu[idx]), 6),
                    "aux_gate_mean": round(float(query_gate_cpu[idx]), 6),
                    "post_reasoning_gate": round(float(post_reasoning_gate_cpu[idx]), 6),
                })

        progress.set_postfix({"loss": round(running_loss / max(total_rows, 1), 5)})

    metrics = {
        "loss": round(running_loss / max(total_rows, 1), 6),
    }
    metrics.update(
        compute_metrics(
            gold_ids=gold_ids,
            pred_ids=pred_ids,
            family_labels=family_labels,
            question_style_labels=question_style_labels,
            difficulty_labels=difficulty_labels,
            ambiguity_labels=ambiguity_labels,
            signal_gap_labels=signal_gap_labels,
            region_pair_labels=region_pair_labels,
        )
    )

    attention_summary = summarize_attention(
        attention_rows=attention_rows,
        family_labels=family_labels,
        ambiguity_labels=ambiguity_labels,
        signal_gap_labels=signal_gap_labels,
    )
    query_context_summary = summarize_query_context(
        query_gate_rows=query_gate_rows,
        family_labels=family_labels,
        ambiguity_labels=ambiguity_labels,
        signal_gap_labels=signal_gap_labels,
    )
    post_reasoning_summary = summarize_post_reasoning(
        attended_uncertainty_rows=attended_uncertainty_rows,
        post_reasoning_gate_rows=post_reasoning_gate_rows,
        family_labels=family_labels,
        ambiguity_labels=ambiguity_labels,
        signal_gap_labels=signal_gap_labels,
        region_pair_labels=region_pair_labels,
    )

    return metrics, prediction_rows, attention_summary, query_context_summary, post_reasoning_summary


def checkpoint_payload(
    model,
    optimizer,
    scheduler,
    scaler,
    epoch,
    history,
    best_val_macro_f1,
    best_epoch,
):
    return {
        "saved_at": now_string(),
        "epoch": int(epoch),
        "best_val_macro_f1": float(best_val_macro_f1),
        "best_epoch": int(best_epoch),
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
        "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
        "history": history,
        "config_path": str(PHASE4_CONFIG_PATH),
        "dataset_release_name": DATASET_RELEASE_NAME,
        "phase": "Phase 4 - BTUMQA Pure Post-Reasoning QGCA Training",
    }


def save_checkpoint(path, model, optimizer, scheduler, scaler, epoch, history, best_val_macro_f1, best_epoch):
    payload = checkpoint_payload(
        model=model,
        optimizer=optimizer,
        scheduler=scheduler,
        scaler=scaler,
        epoch=epoch,
        history=history,
        best_val_macro_f1=best_val_macro_f1,
        best_epoch=best_epoch,
    )
    atomic_torch_save(payload, path)


def maybe_resume_checkpoint(model, optimizer, scheduler, scaler):
    if not RESUME_TRAINING or not LAST_CKPT_PATH.exists():
        return 0, -1.0, -1, []

    payload = safe_torch_load(LAST_CKPT_PATH, map_location="cpu")

    model.load_state_dict(payload["model_state_dict"])
    optimizer.load_state_dict(payload["optimizer_state_dict"])

    if scheduler is not None and payload.get("scheduler_state_dict") is not None:
        scheduler.load_state_dict(payload["scheduler_state_dict"])

    if scaler is not None and payload.get("scaler_state_dict") is not None:
        scaler.load_state_dict(payload["scaler_state_dict"])

    start_epoch = int(payload["epoch"]) + 1
    best_val_macro_f1 = float(payload.get("best_val_macro_f1", -1.0))
    best_epoch = int(payload.get("best_epoch", -1))
    history = payload.get("history", [])

    print(f"Resumed from checkpoint: {LAST_CKPT_PATH}")
    print(f"  next_epoch={start_epoch}")
    print(f"  best_val_macro_f1={best_val_macro_f1:.6f}")
    print(f"  best_epoch={best_epoch}")

    return start_epoch, best_val_macro_f1, best_epoch, history


print("Training and evaluation helpers ready.")


Training and evaluation helpers ready.


# Inspect Phase 4 BTUMQA QA-PRUGTM Hybrid Seed Status


In [ ]:
SEED_STATUS_ROWS = []

for run_seed in RUN_SEEDS:
    context = build_seed_paths(run_seed)
    SEED_STATUS_ROWS.append(
        {
            "run_seed": int(run_seed),
            "seed_tag": context["SEED_TAG"],
            "phase4_dir": str(context["PHASE4_DIR"]),
            "training_complete": bool(training_outputs_complete(context)),
            "final_eval_complete": bool(final_eval_outputs_complete(context)),
            "final_eval_fresh": bool(final_eval_is_fresh(context)),
            "report_exists": bool(phase4_report_complete(context)),
            "full_seed_complete": bool(full_seed_outputs_complete(context)),
        }
    )

seed_status_df = pd.DataFrame(SEED_STATUS_ROWS).sort_values("run_seed").reset_index(drop=True)
print("Seed status before execution:")
print(seed_status_df.to_string(index=False))


Seed status before execution:
 run_seed  seed_tag                                                                                                                                                                                                   phase4_dir  training_complete  final_eval_complete  final_eval_fresh  report_exists  full_seed_complete
       42   seed_42   /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_4_qgca_training/biasless_metadata_free_training/adaptive_prugtm_qgca_btumqa_four_seeds/seed_42               True                 True              True           True                True
     1337 seed_1337 /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_4_qgca_training/biasless_metadata_free_training/adaptive_prugtm_qgca_btumqa_four_seeds/seed_1337               True                 True              True           True                True
     2025 seed_2

# Execute Phase 4 BTUMQA QA-PRUGTM Hybrid Multi-Seed Training


In [ ]:
RUN_EXECUTION_SUMMARY = []

for run_seed in RUN_SEEDS:
    activate_seed_context(run_seed)
    seed_everything(run_seed)

    print("\n" + "=" * 100)
    print(f"Phase 4 BTUMQA QA-PRUGTM Hybrid | {SEED_TAG}")
    print("Run directory:", PHASE4_DIR)
    print("Training complete before run:", training_outputs_complete())
    print("Final eval complete before run:", final_eval_outputs_complete())
    print("Final eval fresh before run:", final_eval_is_fresh())
    print("Report exists before run:", phase4_report_complete())


    seed_is_trainable = int(run_seed) in TRAINABLE_SEEDS
    if not seed_is_trainable:
        if not full_seed_outputs_complete():
            raise RuntimeError(
                f"{SEED_TAG} is a copied legacy seed but its four-seeds output is incomplete: {PHASE4_DIR}"
            )

        report_preview = read_json(PHASE4_REPORT_PATH, default={})
        validation_block = report_preview.get("validation", {})
        test_block = report_preview.get("test", {})
        training_block = report_preview.get("training", {})

        RUN_EXECUTION_SUMMARY.append(
            {
                "run_seed": int(run_seed),
                "seed_tag": SEED_TAG,
                "phase4_dir": str(PHASE4_DIR),
                "training_status": "copied_legacy_skipped",
                "final_eval_status": "copied_legacy_skipped",
                "best_epoch": training_block.get("best_epoch"),
                "best_val_macro_f1": training_block.get("best_val_macro_f1"),
                "val_accuracy": validation_block.get("accuracy"),
                "test_accuracy": test_block.get("accuracy"),
                "test_macro_f1": test_block.get("macro_f1"),
                "val_predictions_path": str(VAL_PREDICTIONS_PATH),
                "test_predictions_path": str(TEST_PREDICTIONS_PATH),
                "report_path": str(PHASE4_REPORT_PATH),
            "input_policy_path": str(INPUT_POLICY_PATH),
            }
        )
        print(f"{SEED_TAG} is copied legacy output. Skipping config rewrite, training, and final evaluation.")
        continue

    input_policy_report = build_input_policy_report(run_seed=run_seed)

    phase4_config = {
        "created_at": now_string(),
        "phase": "Phase 4 - BTUMQA QA-PRUGTM Hybrid Clean Adaptive Uncertainty QGCA Training",
        "dataset_release_name": DATASET_RELEASE_NAME,
        "phase3a_release_key": PHASE3A_RELEASE_KEY,
        "run_seed": int(run_seed),
        "seed_tag": SEED_TAG,
        "phase4_parent_dir": str(PHASE4_PARENT_DIR),
        "input_policy_path": str(INPUT_POLICY_PATH),
        "input_policy": input_policy_report,
        "model_variant": PHASE4_MODEL_VARIANT,
        "main_method_name": PHASE4_METHOD_NAME,
        "fusion_type": PHASE4_FUSION_TYPE,
        "main_claim": (
            "QGCA reasons over original BTUMQA visual evidence while a question-region gate "
            "adaptively controls visual-derived region_aux uncertainty guidance before attention."
        ),
        "project_drive_dir": str(PROJECT_DRIVE_DIR),
        "phase2c_save_dir": str(PHASE2C_SAVE_DIR),
        "phase3a_dir": str(PHASE3A_DIR),
        "phase3b_dir": str(PHASE3B_DIR),
        "phase4_dir": str(PHASE4_DIR),
        "train_csv_path": str(TRAIN_CSV_PATH),
        "val_csv_path": str(VAL_CSV_PATH),
        "test_csv_path": str(TEST_CSV_PATH),
        "embeddings_train_path": str(EMBEDDINGS_TRAIN_PATH),
        "embeddings_val_path": str(EMBEDDINGS_VAL_PATH),
        "embeddings_test_path": str(EMBEDDINGS_TEST_PATH),
        "answer_vocab_path": str(ANSWER_VOCAB_PATH),
        "question_family_vocab_path": str(QUESTION_FAMILY_VOCAB_PATH),
        "question_style_vocab_path": str(QUESTION_STYLE_VOCAB_PATH),
        "difficulty_level_vocab_path": str(DIFFICULTY_LEVEL_VOCAB_PATH),
        "ambiguity_flag_vocab_path": str(AMBIGUITY_FLAG_VOCAB_PATH),
        "signal_gap_bucket_vocab_path": str(SIGNAL_GAP_BUCKET_VOCAB_PATH),
        "region_target_vocab_path": str(REGION_TARGET_VOCAB_PATH),
        "phase3b_config_path": str(PHASE3B_CONFIG_PATH),
        "phase3b_report_path": str(PHASE3B_REPORT_PATH),
        "text_embedding_key": TEXT_EMBED_KEY,
        "text_embed_dim": TEXT_EMBED_DIM,
        "visual_token_dim": VISUAL_TOKEN_DIM,
        "model_dim": MODEL_DIM,
        "aux_feature_dim": AUX_FEATURE_DIM,
        "num_regions": NUM_VISUAL_TOKENS,
        "region_names": REGION_NAMES,
        "num_answers": len(answer_vocab),
        "num_question_families": len(question_family_vocab),
        "num_attention_heads": NUM_ATTENTION_HEADS,
        "aux_hidden_dim": AUX_HIDDEN_DIM,
        "classifier_hidden_dim": CLASSIFIER_HIDDEN_DIM,
        "context_embed_dim": CONTEXT_EMBED_DIM,
        "context_hidden_dim": CONTEXT_HIDDEN_DIM,
        "selected_context_fields": SELECTED_CONTEXT_FIELDS,
        "adaptive_aux_gate_init_probability": AUX_GATE_INIT_PROB,
        "adaptive_aux_gate_fusion": "visual_proj(visual_tokens) + aux_gate * aux_encoder(region_aux) + region_embeddings",
        "dropout": DROPOUT,
        "train_batch_size": TRAIN_BATCH_SIZE,
        "eval_batch_size": EVAL_BATCH_SIZE,
        "num_workers": NUM_WORKERS,
        "pin_memory": PIN_MEMORY,
        "max_epochs": MAX_EPOCHS,
        "min_epochs_before_early_stop": MIN_EPOCHS_BEFORE_EARLY_STOP,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "label_smoothing": LABEL_SMOOTHING,
        "grad_clip_norm": GRAD_CLIP_NORM,
        "use_class_weights": USE_CLASS_WEIGHTS,
        "use_amp": USE_AMP,
        "resume_training": RESUME_TRAINING,
        "strict_phase2c_alignment": STRICT_PHASE2C_ALIGNMENT,
        "visual_cache_dtype": str(VISUAL_CACHE_DTYPE),
        "reuse_saved_visual_cache": REUSE_SAVED_VISUAL_CACHE,
        "force_rebuild_visual_cache": FORCE_REBUILD_VISUAL_CACHE,
        "skip_train_if_already_done": SKIP_TRAIN_IF_ALREADY_DONE,
        "skip_final_eval_if_already_done": SKIP_FINAL_EVAL_IF_ALREADY_DONE,
        "force_rerun_final_eval": FORCE_RERUN_FINAL_EVAL,
        "visual_cache_tensor_path": str(VISUAL_CACHE_TENSOR_PATH),
        "visual_cache_metadata_path": str(VISUAL_CACHE_METADATA_PATH),
        "expected_split_rows": {
            "train": int(len(df_train)),
            "val": int(len(df_val)),
            "test": int(len(df_test)),
        },
        "phase4_outputs": {
            "best_checkpoint": str(BEST_CKPT_PATH),
            "last_checkpoint": str(LAST_CKPT_PATH),
            "history_path": str(PHASE4_HISTORY_PATH),
            "report_path": str(PHASE4_REPORT_PATH),
            "val_metrics_path": str(VAL_METRICS_PATH),
            "test_metrics_path": str(TEST_METRICS_PATH),
            "val_attention_path": str(VAL_ATTENTION_PATH),
            "test_attention_path": str(TEST_ATTENTION_PATH),
            "val_query_context_path": str(VAL_QUERY_CONTEXT_PATH),
            "test_query_context_path": str(TEST_QUERY_CONTEXT_PATH),
            "val_post_reasoning_path": str(VAL_POST_REASONING_PATH),
            "test_post_reasoning_path": str(TEST_POST_REASONING_PATH),
            "final_eval_summary_path": str(FINAL_EVAL_SUMMARY_PATH),
            "val_predictions_path": str(VAL_PREDICTIONS_PATH),
            "test_predictions_path": str(TEST_PREDICTIONS_PATH),
            "training_done_marker": str(TRAINING_DONE_PATH),
            "final_eval_done_marker": str(FINAL_EVAL_DONE_PATH),
        },
    }

    atomic_write_json(INPUT_POLICY_PATH, input_policy_report)
    atomic_write_json(PHASE4_CONFIG_PATH, phase4_config)
    print(json.dumps(input_policy_report, indent=2))
    print("Input policy saved:", INPUT_POLICY_PATH)
    print(json.dumps(phase4_config, indent=2))
    print("Phase 4 BTUMQA QA-PRUGTM hybrid config saved:", PHASE4_CONFIG_PATH)


    model = Phase4SingleStageBTUMQAQAPRUGTMHybridModel(
        text_embed_dim=TEXT_EMBED_DIM,
        visual_token_dim=VISUAL_TOKEN_DIM,
        model_dim=MODEL_DIM,
        aux_feature_dim=AUX_FEATURE_DIM,
        num_regions=NUM_VISUAL_TOKENS,
        num_attention_heads=NUM_ATTENTION_HEADS,
        classifier_hidden_dim=CLASSIFIER_HIDDEN_DIM,
        aux_hidden_dim=AUX_HIDDEN_DIM,
        dropout=DROPOUT,
        num_answers=len(answer_vocab),
        num_question_styles=len(question_style_vocab),
        context_embed_dim=CONTEXT_EMBED_DIM,
        context_hidden_dim=CONTEXT_HIDDEN_DIM,
    ).to(DEVICE)

    forward_signature_check = assert_clean_model_forward_signature(model)
    input_policy_report["runtime_forward_signature_check"] = forward_signature_check
    atomic_write_json(INPUT_POLICY_PATH, input_policy_report)
    print("Clean model forward signature check:", forward_signature_check)

    criterion = nn.CrossEntropyLoss(
        weight=class_weights.to(DEVICE) if USE_CLASS_WEIGHTS else None,
        label_smoothing=LABEL_SMOOTHING,
    )

    optimizer = AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
    )

    scaler = GradScaler(enabled=USE_AMP)

    sample_batch = next(iter(train_loader))
    sample_batch_moved = move_batch_to_device(sample_batch)
    with torch.no_grad():
        sample_outputs = model(
            question_embedding=sample_batch_moved["question_embedding"],
            visual_tokens=sample_batch_moved["visual_tokens"],
            region_aux=sample_batch_moved["region_aux"],
        )
    print("Smoke test logits shape:", tuple(sample_outputs["logits"].shape))
    print("Smoke test attention shape:", tuple(sample_outputs["attention_weights"].shape))
    print("Smoke test attended uncertainty shape:", tuple(sample_outputs["attended_uncertainty"].shape))
    print("Smoke test post reasoning gate shape:", tuple(sample_outputs["post_reasoning_gate"].shape))
    print("Smoke test aux gate shape:", tuple(sample_outputs["aux_gate"].shape))
    print("Smoke test aux gate range:", float(sample_outputs["aux_gate"].min().item()), float(sample_outputs["aux_gate"].max().item()))
    print("Smoke test mean aux gate shape:", tuple(sample_outputs["query_gate_mean"].shape))
    assert tuple(sample_outputs["aux_gate"].shape) == (sample_batch_moved["question_embedding"].shape[0], NUM_VISUAL_TOKENS, 1)
    assert float(sample_outputs["aux_gate"].min().item()) >= 0.0
    assert float(sample_outputs["aux_gate"].max().item()) <= 1.0
    assert tuple(sample_outputs["logits"].shape) == (sample_batch_moved["question_embedding"].shape[0], len(answer_vocab))
    del sample_batch, sample_batch_moved, sample_outputs
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    training_already_done = SKIP_TRAIN_IF_ALREADY_DONE and training_outputs_complete()

    if training_already_done:
        done_payload = read_json(TRAINING_DONE_PATH, default={})
        print("Phase 4 BTUMQA QA-PRUGTM hybrid training artifacts already exist. Skipping retraining.")
        if done_payload:
            print(json.dumps(done_payload, indent=2))
    else:
        start_epoch, best_val_macro_f1, best_epoch, history = maybe_resume_checkpoint(
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=scaler,
        )

        epochs_without_improvement = 0
        if len(history) > 0:
            last_best_seen = max(float(item["val"]["macro_f1"]) for item in history)
            if abs(last_best_seen - best_val_macro_f1) < 1e-9:
                best_epoch = max(
                    int(item["epoch"])
                    for item in history
                    if abs(float(item["val"]["macro_f1"]) - best_val_macro_f1) < 1e-9
                )
                epochs_without_improvement = max(0, len(history) - 1 - best_epoch)

        if start_epoch >= MAX_EPOCHS:
            print("Training already reached or exceeded MAX_EPOCHS for this BTUMQA QA-PRUGTM hybrid run.")
        else:
            run_started = time.time()

            for epoch in range(start_epoch, MAX_EPOCHS):
                epoch_started = time.time()
                current_lr = float(optimizer.param_groups[0]["lr"])
                print(f"\nEpoch {epoch + 1}/{MAX_EPOCHS} | lr={current_lr:.8f}")

                train_metrics = train_one_epoch(
                    model=model,
                    loader=train_loader,
                    optimizer=optimizer,
                    criterion=criterion,
                    scaler=scaler,
                    epoch_index=epoch,
                )

                val_metrics, _, _, _, _ = evaluate_model(
                    model=model,
                    loader=val_loader,
                    criterion=criterion,
                    split_name="val",
                    return_predictions=False,
                )

                scheduler.step(val_metrics["macro_f1"])

                epoch_record = {
                    "epoch": int(epoch),
                    "finished_at": now_string(),
                    "elapsed_minutes": round((time.time() - epoch_started) / 60, 2),
                    "learning_rate": round(current_lr, 10),
                    "train": train_metrics,
                    "val": val_metrics,
                }
                history.append(epoch_record)

                improved = val_metrics["macro_f1"] > best_val_macro_f1 + 1e-6
                if improved:
                    best_val_macro_f1 = float(val_metrics["macro_f1"])
                    best_epoch = int(epoch)
                    epochs_without_improvement = 0

                    save_checkpoint(
                        path=BEST_CKPT_PATH,
                        model=model,
                        optimizer=optimizer,
                        scheduler=scheduler,
                        scaler=scaler,
                        epoch=epoch,
                        history=history,
                        best_val_macro_f1=best_val_macro_f1,
                        best_epoch=best_epoch,
                    )
                    print(f"  New best checkpoint saved: {BEST_CKPT_PATH}")
                else:
                    epochs_without_improvement += 1

                save_checkpoint(
                    path=LAST_CKPT_PATH,
                    model=model,
                    optimizer=optimizer,
                    scheduler=scheduler,
                    scaler=scaler,
                    epoch=epoch,
                    history=history,
                    best_val_macro_f1=best_val_macro_f1,
                    best_epoch=best_epoch,
                )

                history_payload = {
                    "updated_at": now_string(),
                    "dataset_release_name": DATASET_RELEASE_NAME,
                    "model_variant": PHASE4_MODEL_VARIANT,
                    "best_val_macro_f1": float(best_val_macro_f1),
                    "best_epoch": int(best_epoch),
                    "history": history,
                }
                atomic_write_json(PHASE4_HISTORY_PATH, history_payload)

                print(
                    "  train_loss={:.6f} | val_loss={:.6f} | val_acc={:.6f} | val_macro_f1={:.6f}".format(
                        train_metrics["loss"],
                        val_metrics["loss"],
                        val_metrics["accuracy"],
                        val_metrics["macro_f1"],
                    )
                )

                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                gc.collect()

                if (epoch + 1) >= MIN_EPOCHS_BEFORE_EARLY_STOP and epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
                    print(f"Early stopping triggered at epoch {epoch + 1}")
                    break

            training_elapsed_minutes = round((time.time() - run_started) / 60, 2)
            print("\nPhase 4 BTUMQA QA-PRUGTM hybrid training finished.")
            print("Training elapsed minutes:", training_elapsed_minutes)

        training_done_payload = {
            "finished_at": now_string(),
            "phase": "Phase 4 - BTUMQA Hybrid Post-Reasoning QGCA Training",
            "dataset_release_name": DATASET_RELEASE_NAME,
            "model_variant": PHASE4_MODEL_VARIANT,
            "run_seed": int(run_seed),
            "seed_tag": SEED_TAG,
            "best_checkpoint": str(BEST_CKPT_PATH) if BEST_CKPT_PATH.exists() else None,
            "last_checkpoint": str(LAST_CKPT_PATH) if LAST_CKPT_PATH.exists() else None,
            "training_history_path": str(PHASE4_HISTORY_PATH),
            "best_epoch": int(best_epoch) if best_epoch >= 0 else None,
            "best_val_macro_f1": float(best_val_macro_f1) if best_val_macro_f1 >= 0 else None,
        }
        atomic_write_json(TRAINING_DONE_PATH, training_done_payload)
        print(json.dumps(training_done_payload, indent=2))
        print("Training done marker saved:", TRAINING_DONE_PATH)


    final_eval_skipped = (
        SKIP_FINAL_EVAL_IF_ALREADY_DONE
        and not FORCE_RERUN_FINAL_EVAL
        and final_eval_is_fresh()
    )

    if final_eval_skipped:
        print("Final validation/test outputs already exist and are fresh. Skipping re-evaluation.")
        final_eval_summary = read_json(FINAL_EVAL_SUMMARY_PATH, default={})
        print(json.dumps(final_eval_summary, indent=2))
    else:
        if not BEST_CKPT_PATH.exists():
            raise FileNotFoundError(f"Best checkpoint not found: {BEST_CKPT_PATH}")

        best_payload = safe_torch_load(BEST_CKPT_PATH, map_location="cpu")
        model.load_state_dict(best_payload["model_state_dict"])
        model = model.to(DEVICE)
        model.eval()

        val_metrics, val_prediction_rows, val_attention_summary, val_query_context_summary, val_post_reasoning_summary = evaluate_model(
            model=model,
            loader=val_loader,
            criterion=criterion,
            split_name="val",
            return_predictions=True,
        )

        test_metrics, test_prediction_rows, test_attention_summary, test_query_context_summary, test_post_reasoning_summary = evaluate_model(
            model=model,
            loader=test_loader,
            criterion=criterion,
            split_name="test",
            return_predictions=True,
        )

        prediction_fieldnames = [
            "split",
            "qa_id",
            "unique_id",
            "question_family",
            "question_style",
            "difficulty_level",
            "ambiguity_flag",
            "signal_gap_bucket",
            "region_target_primary",
            "region_target_secondary",
            "region_target_pair",
            "gold_answer",
            "predicted_answer",
            "confidence",
            "attended_uncertainty",
            "aux_gate_mean",
            "post_reasoning_gate",
        ]

        atomic_write_csv(VAL_PREDICTIONS_PATH, val_prediction_rows, prediction_fieldnames)
        atomic_write_csv(TEST_PREDICTIONS_PATH, test_prediction_rows, prediction_fieldnames)

        atomic_write_json(VAL_METRICS_PATH, val_metrics)
        atomic_write_json(TEST_METRICS_PATH, test_metrics)
        atomic_write_json(VAL_ATTENTION_PATH, val_attention_summary)
        atomic_write_json(TEST_ATTENTION_PATH, test_attention_summary)
        atomic_write_json(VAL_QUERY_CONTEXT_PATH, val_query_context_summary)
        atomic_write_json(TEST_QUERY_CONTEXT_PATH, test_query_context_summary)
        atomic_write_json(VAL_POST_REASONING_PATH, val_post_reasoning_summary)
        atomic_write_json(TEST_POST_REASONING_PATH, test_post_reasoning_summary)

        final_eval_summary = {
            "finished_at": now_string(),
            "phase": "Phase 4 - Single-Stage BTUMQA Hybrid Post-Reasoning QGCA Final Validation/Test Inference",
            "model_variant": PHASE4_MODEL_VARIANT,
            "run_seed": int(run_seed),
            "seed_tag": SEED_TAG,
            "loaded_checkpoint": str(BEST_CKPT_PATH),
            "best_epoch": int(best_payload["best_epoch"]),
            "best_val_macro_f1": float(best_payload["best_val_macro_f1"]),
            "val_metrics": val_metrics,
            "test_metrics": test_metrics,
            "val_predictions_path": str(VAL_PREDICTIONS_PATH),
            "test_predictions_path": str(TEST_PREDICTIONS_PATH),
            "val_attention_path": str(VAL_ATTENTION_PATH),
            "test_attention_path": str(TEST_ATTENTION_PATH),
            "val_query_context_path": str(VAL_QUERY_CONTEXT_PATH),
            "test_query_context_path": str(TEST_QUERY_CONTEXT_PATH),
            "val_post_reasoning_path": str(VAL_POST_REASONING_PATH),
            "test_post_reasoning_path": str(TEST_POST_REASONING_PATH),
        }

        atomic_write_json(FINAL_EVAL_SUMMARY_PATH, final_eval_summary)
        atomic_write_json(
            FINAL_EVAL_DONE_PATH,
            {
                "finished_at": now_string(),
                "phase": "Phase 4 - Single-Stage BTUMQA Hybrid Post-Reasoning QGCA Final Validation/Test Inference",
                "dataset_release_name": DATASET_RELEASE_NAME,
                "model_variant": PHASE4_MODEL_VARIANT,
                "run_seed": int(run_seed),
                "seed_tag": SEED_TAG,
                "best_checkpoint": str(BEST_CKPT_PATH),
                "final_eval_summary_path": str(FINAL_EVAL_SUMMARY_PATH),
            },
        )

        print(json.dumps(final_eval_summary, indent=2))
        print("Final validation/test inference complete.")
        print("Final eval summary saved:", FINAL_EVAL_SUMMARY_PATH)
        print("Final eval done marker saved:", FINAL_EVAL_DONE_PATH)


    if not BEST_CKPT_PATH.exists():
        raise FileNotFoundError(f"Missing best checkpoint: {BEST_CKPT_PATH}")

    best_payload = safe_torch_load(BEST_CKPT_PATH, map_location="cpu")
    val_metrics = read_json(VAL_METRICS_PATH, default={})
    test_metrics = read_json(TEST_METRICS_PATH, default={})
    final_eval_summary = read_json(FINAL_EVAL_SUMMARY_PATH, default={})
    training_done_payload = read_json(TRAINING_DONE_PATH, default={})
    final_eval_done_payload = read_json(FINAL_EVAL_DONE_PATH, default={})
    val_query_context_summary = read_json(VAL_QUERY_CONTEXT_PATH, default={})
    test_query_context_summary = read_json(TEST_QUERY_CONTEXT_PATH, default={})
    val_post_reasoning_summary = read_json(VAL_POST_REASONING_PATH, default={})
    test_post_reasoning_summary = read_json(TEST_POST_REASONING_PATH, default={})

    phase4_report = {
        "finished_at": now_string(),
        "phase": "Phase 4 - BTUMQA QA-PRUGTM Hybrid Clean Adaptive Uncertainty QGCA Training",
        "status": "complete",
        "dataset_release_name": DATASET_RELEASE_NAME,
        "phase3a_release_key": PHASE3A_RELEASE_KEY,
        "run_seed": int(run_seed),
        "seed_tag": SEED_TAG,
        "phase4_parent_dir": str(PHASE4_PARENT_DIR),
        "input_policy": read_json(INPUT_POLICY_PATH, default=input_policy_report),
        "model_variant": PHASE4_MODEL_VARIANT,
        "main_method_name": PHASE4_METHOD_NAME,
        "main_claim": (
            "Question-adaptive uncertainty guidance preserves original visual evidence, gates "
            "visual-derived region_aux before QGCA attention, and retains post-reasoning reliability gating."
        ),
        "text_input_source": {
            "phase": "Phase 3B",
            "embedding_key": TEXT_EMBED_KEY,
            "embedding_dim": TEXT_EMBED_DIM,
            "source_dir": str(PHASE3B_DIR),
        },
        "visual_input_source": {
            "phase": "Phase 2C",
            "token_field": "original_tokens",
            "token_shape_per_sample": [NUM_VISUAL_TOKENS, VISUAL_TOKEN_DIM],
            "region_names": REGION_NAMES,
            "source_dir": str(PHASE2C_SAVE_DIR),
        },
        "uncertainty_usage": {
            "pre_attention_token_suppression": False,
            "pre_attention_aux_feature_injection": True,
            "query_context_refinement_enabled": False,
            "question_region_aux_gate_enabled": True,
            "question_region_aux_gate_init_probability": AUX_GATE_INIT_PROB,
            "question_region_aux_gate_fusion": "visual_proj(visual_tokens) + aux_gate * aux_encoder(region_aux) + region_embeddings",
            "query_context_selected_fields": SELECTED_CONTEXT_FIELDS,
            "clean_metadata_policy": "Question-visible metadata/context labels are retained only for reporting, not passed to the model.",
            "post_reasoning_gate": "1 - attention-weighted regional uncertainty",
            "regional_uncertainty_source": "region_aux[..., 0]",
        },
        "model_architecture": {
            "question_projection": f"{TEXT_EMBED_DIM} -> {MODEL_DIM}",
            "visual_projection": f"{VISUAL_TOKEN_DIM} -> {MODEL_DIM}",
            "aux_encoder": f"{AUX_FEATURE_DIM} -> {AUX_HIDDEN_DIM} -> {MODEL_DIM}",
            "question_region_aux_gate": f"question+region ({MODEL_DIM * 2}) -> {MODEL_DIM} -> 1 sigmoid",
            "aux_features_available": ["uncertainty", "modulation_weight", "region_present", "relative_area"],
            "cross_attention": {
                "model_dim": MODEL_DIM,
                "num_heads": NUM_ATTENTION_HEADS,
                "num_regions": NUM_VISUAL_TOKENS,
            },
            "fusion_type": PHASE4_FUSION_TYPE,
            "classifier_hidden_dim": CLASSIFIER_HIDDEN_DIM,
            "dropout": DROPOUT,
            "num_answers": len(answer_vocab),
        },
        "training": {
            "device": str(DEVICE),
            "run_seed": int(run_seed),
            "seed_tag": SEED_TAG,
            "train_batch_size": TRAIN_BATCH_SIZE,
            "eval_batch_size": EVAL_BATCH_SIZE,
            "max_epochs": MAX_EPOCHS,
            "best_epoch": int(best_payload["best_epoch"]),
            "best_val_macro_f1": float(best_payload["best_val_macro_f1"]),
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "label_smoothing": LABEL_SMOOTHING,
            "use_class_weights": USE_CLASS_WEIGHTS,
            "use_amp": USE_AMP,
            "resume_training": RESUME_TRAINING,
            "skip_train_if_already_done": SKIP_TRAIN_IF_ALREADY_DONE,
            "skip_final_eval_if_already_done": SKIP_FINAL_EVAL_IF_ALREADY_DONE,
            "training_history_path": str(PHASE4_HISTORY_PATH),
        },
        "visual_cache_summary": read_json(VISUAL_CACHE_METADATA_PATH, default=visual_cache_summary),
        "training_done_payload": training_done_payload,
        "final_eval_done_payload": final_eval_done_payload,
        "final_eval_summary": final_eval_summary,
        "validation": val_metrics,
        "test": test_metrics,
        "validation_query_context_summary": val_query_context_summary,
        "test_query_context_summary": test_query_context_summary,
        "validation_post_reasoning_summary": val_post_reasoning_summary,
        "test_post_reasoning_summary": test_post_reasoning_summary,
        "outputs": {
            "config": str(PHASE4_CONFIG_PATH),
            "input_policy": str(INPUT_POLICY_PATH),
            "history": str(PHASE4_HISTORY_PATH),
            "report": str(PHASE4_REPORT_PATH),
            "best_checkpoint": str(BEST_CKPT_PATH),
            "last_checkpoint": str(LAST_CKPT_PATH),
            "visual_cache_tensor": str(VISUAL_CACHE_TENSOR_PATH),
            "visual_cache_metadata": str(VISUAL_CACHE_METADATA_PATH),
            "val_metrics": str(VAL_METRICS_PATH),
            "test_metrics": str(TEST_METRICS_PATH),
            "val_attention": str(VAL_ATTENTION_PATH),
            "test_attention": str(TEST_ATTENTION_PATH),
            "val_query_context_summary": str(VAL_QUERY_CONTEXT_PATH),
            "test_query_context_summary": str(TEST_QUERY_CONTEXT_PATH),
            "val_post_reasoning_summary": str(VAL_POST_REASONING_PATH),
            "test_post_reasoning_summary": str(TEST_POST_REASONING_PATH),
            "final_eval_summary": str(FINAL_EVAL_SUMMARY_PATH),
            "val_predictions": str(VAL_PREDICTIONS_PATH),
            "test_predictions": str(TEST_PREDICTIONS_PATH),
            "training_done_marker": str(TRAINING_DONE_PATH),
            "final_eval_done_marker": str(FINAL_EVAL_DONE_PATH),
        },
        "next_step": (
            "Phase 5 / Q-CUR - calibrate confidence and reliability on top of this hybrid "
            "post-reasoning uncertainty backbone using ECE, Brier, NLL, AURC, and selective accuracy."
        ),
    }

    atomic_write_json(PHASE4_REPORT_PATH, phase4_report)

    print(json.dumps(phase4_report, indent=2))
    print("Final BTUMQA Phase 4 QA-PRUGTM hybrid report saved:", PHASE4_REPORT_PATH)


    RUN_EXECUTION_SUMMARY.append(
        {
            "run_seed": int(run_seed),
            "seed_tag": SEED_TAG,
            "phase4_dir": str(PHASE4_DIR),
            "training_status": "skipped_existing" if training_already_done else "trained_or_resumed",
            "final_eval_status": "skipped_fresh" if final_eval_skipped else "evaluated",
            "best_epoch": int(best_payload["best_epoch"]) + 1,
            "best_val_macro_f1": float(best_payload["best_val_macro_f1"]),
            "val_accuracy": float(val_metrics.get("accuracy", 0.0)),
            "test_accuracy": float(test_metrics.get("accuracy", 0.0)),
            "test_macro_f1": float(test_metrics.get("macro_f1", 0.0)),
            "val_predictions_path": str(VAL_PREDICTIONS_PATH),
            "test_predictions_path": str(TEST_PREDICTIONS_PATH),
            "report_path": str(PHASE4_REPORT_PATH),
        }
    )

    del model, criterion, optimizer, scheduler, scaler
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()



Phase 4 BTUMQA QA-PRUGTM Hybrid | seed_42
Run directory: /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_4_qgca_training/biasless_metadata_free_training/adaptive_prugtm_qgca_btumqa_four_seeds/seed_42
Training complete before run: True
Final eval complete before run: True
Final eval fresh before run: True
Report exists before run: True
{
  "created_at": "2026-05-12 21:14:54",
  "phase": "Phase 4 clean-metadata BTUMQA training input policy",
  "dataset_release_name": "BTUMQA-225K",
  "model_variant": "btumqa_225k_qgca_qadp_prugtm_hybrid_adaptive_uncertainty_clean_metadata",
  "run_seed": 42,
  "seed_tag": "seed_42",
  "policy_name": "strict_no_explicit_metadata_context",
  "phase4_parent_dir": "/content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_4_qgca_training/biasless_metadata_free_training/adaptive_prugtm_qgca_btumqa_four_seeds",
  "input_policy_path": "/content/drive/M

/tmp/ipykernel_19570/1730467212.py:203: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=USE_AMP)


Streaming output truncated to the last 5000 lines.
          "count": 2700,
          "macro_f1": 0.947948120920926
        },
        "more_uncertain_region": {
          "accuracy": 0.9522222222222222,
          "count": 2700,
          "macro_f1": 0.9526693844457947
        },
        "reliability_gap_bucket": {
          "accuracy": 0.8240740740740741,
          "count": 2700,
          "macro_f1": 0.7798741499933343
        },
        "safe_region_for_reasoning": {
          "accuracy": 0.99,
          "count": 2700,
          "macro_f1": 0.8782683339249093
        },
        "uncertainty_consistency_check": {
          "accuracy": 0.9122703023117961,
          "count": 3374,
          "macro_f1": 0.8047064252834454
        },
        "uncertainty_gap_bucket": {
          "accuracy": 0.8174074074074074,
          "count": 2700,
          "macro_f1": 0.7923602284594483
        }
      },
      "per_question_style": {
        "ambiguity_sensitive": {
          "accuracy": 0.90318118

Train Epoch 1:   0%|          | 0/1641 [00:00<?, ?it/s]

Evaluate val:   0%|          | 0/176 [00:00<?, ?it/s]

...

Evaluate val:   0%|          | 0/176 [00:00<?, ?it/s]

  train_loss=0.817306 | val_loss=0.860514 | val_acc=0.895289 | val_macro_f1=0.895542

Phase 4 BTUMQA QA-PRUGTM hybrid training finished.
Training elapsed minutes: 29.3
{
  "finished_at": "2026-05-12 22:12:04",
  "phase": "Phase 4 - BTUMQA Hybrid Post-Reasoning QGCA Training",
  "dataset_release_name": "BTUMQA-225K",
  "model_variant": "btumqa_225k_qgca_qadp_prugtm_hybrid_adaptive_uncertainty_clean_metadata",
  "run_seed": 3407,
  "seed_tag": "seed_3407",
  "best_checkpoint": "/content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_4_qgca_training/biasless_metadata_free_training/adaptive_prugtm_qgca_btumqa_four_seeds/seed_3407/checkpoints/best_adaptive_prugtm_qgca_model.pt",
  "last_checkpoint": "/content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_4_qgca_training/biasless_metadata_free_training/adaptive_prugtm_qgca_btumqa_four_seeds/seed_3407/checkpoints/last_adaptive_prugtm_qgca_

Evaluate val:   0%|          | 0/176 [00:00<?, ?it/s]

Evaluate test:   0%|          | 0/176 [00:00<?, ?it/s]

{
  "finished_at": "2026-05-12 22:12:12",
  "phase": "Phase 4 - Single-Stage BTUMQA Hybrid Post-Reasoning QGCA Final Validation/Test Inference",
  "model_variant": "btumqa_225k_qgca_qadp_prugtm_hybrid_adaptive_uncertainty_clean_metadata",
  "run_seed": 3407,
  "seed_tag": "seed_3407",
  "loaded_checkpoint": "/content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_4_qgca_training/biasless_metadata_free_training/adaptive_prugtm_qgca_btumqa_four_seeds/seed_3407/checkpoints/best_adaptive_prugtm_qgca_model.pt",
  "best_epoch": 47,
  "best_val_macro_f1": 0.8980155511658059,
  "val_metrics": {
    "loss": 0.855832,
    "accuracy": 0.8972148148148148,
    "macro_f1": 0.8980155511658059,
    "weighted_f1": 0.9001489158991859,
    "per_question_family": {
      "ambiguous_subregion_pair": {
        "count": 3376,
        "accuracy": 0.8166469194312796,
        "macro_f1": 0.7454992607004791
      },
      "confidence_qualified_extent": {
      

# Summarize Phase 4 BTUMQA QA-PRUGTM Hybrid Multi-Seed Results


In [ ]:
summary_df = pd.DataFrame(RUN_EXECUTION_SUMMARY).sort_values("run_seed").reset_index(drop=True)
print("Per-seed validation/test summary:")
print(summary_df[[
    "seed_tag",
    "training_status",
    "final_eval_status",
    "best_epoch",
    "best_val_macro_f1",
    "val_accuracy",
    "test_accuracy",
    "test_macro_f1",
]].round(6).to_string(index=False))

print()
print("Per-seed saved prediction files:")
for row in summary_df.to_dict(orient="records"):
    print(f"{row['seed_tag']}:")
    print("  val_predictions:", row["val_predictions_path"])
    print("  test_predictions:", row["test_predictions_path"])


Per-seed validation/test summary:
 seed_tag    training_status final_eval_status  best_epoch  best_val_macro_f1  val_accuracy  test_accuracy  test_macro_f1
  seed_42   skipped_existing     skipped_fresh          49           0.911607      0.915348       0.911674       0.907967
seed_1337   skipped_existing     skipped_fresh          49           0.924240      0.931526       0.929274       0.920041
seed_2025 trained_or_resumed         evaluated          50           0.911994      0.914281       0.911556       0.909071
seed_3407 trained_or_resumed         evaluated          48           0.898016      0.897215       0.893215       0.893210

Per-seed saved prediction files:
seed_42:
  val_predictions: /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_4_qgca_training/biasless_metadata_free_training/adaptive_prugtm_qgca_btumqa_four_seeds/seed_42/predictions/val_predictions.csv
  test_predictions: /content/drive/MyDrive/AMIR Lab/Researc

# Preview Final BTUMQA Phase 4 QA-PRUGTM Hybrid Reports by Seed


In [ ]:
SEED_REPORT_PREVIEW_ROWS = []

for run_seed in RUN_SEEDS:
    activate_seed_context(run_seed)
    if not PHASE4_REPORT_PATH.exists():
        raise FileNotFoundError(f"Missing final Phase 4 QA-PRUGTM hybrid report for {SEED_TAG}: {PHASE4_REPORT_PATH}")

    report_preview = read_json(PHASE4_REPORT_PATH, {})
    validation_block = report_preview.get("validation", {})
    test_block = report_preview.get("test", {})
    training_block = report_preview.get("training", {})

    SEED_REPORT_PREVIEW_ROWS.append(
        {
            "run_seed": int(run_seed),
            "seed_tag": SEED_TAG,
            "phase4_dir": str(PHASE4_DIR),
            "best_epoch": int(training_block.get("best_epoch", -1)) + 1,
            "best_val_macro_f1": float(training_block.get("best_val_macro_f1", 0.0)),
            "val_accuracy": float(validation_block.get("accuracy", 0.0)),
            "test_accuracy": float(test_block.get("accuracy", 0.0)),
            "test_macro_f1": float(test_block.get("macro_f1", 0.0)),
            "report_path": str(PHASE4_REPORT_PATH),
        }
    )

seed_report_preview_df = pd.DataFrame(SEED_REPORT_PREVIEW_ROWS).sort_values("run_seed").reset_index(drop=True)
print("Per-seed report preview:")
print(seed_report_preview_df[[
    "seed_tag",
    "best_epoch",
    "best_val_macro_f1",
    "val_accuracy",
    "test_accuracy",
    "test_macro_f1",
]].round(6).to_string(index=False))

print()
for row in seed_report_preview_df.to_dict(orient="records"):
    print(f"{row['seed_tag']} report:", row["report_path"])


Per-seed report preview:
 seed_tag  best_epoch  best_val_macro_f1  val_accuracy  test_accuracy  test_macro_f1
  seed_42          49           0.911607      0.915348       0.911674       0.907967
seed_1337          49           0.924240      0.931526       0.929274       0.920041
seed_2025          50           0.911994      0.914281       0.911556       0.909071
seed_3407          48           0.898016      0.897215       0.893215       0.893210

seed_42 report: /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_4_qgca_training/biasless_metadata_free_training/adaptive_prugtm_qgca_btumqa_four_seeds/seed_42/phase4_qa_prugtm_hybrid_training_report.json
seed_1337 report: /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_4_qgca_training/biasless_metadata_free_training/adaptive_prugtm_qgca_btumqa_four_seeds/seed_1337/phase4_qa_prugtm_hybrid_training_report.json
seed_2025 report:

# Quick Inspect Saved BTUMQA Phase 4 QA-PRUGTM Hybrid Artifacts by Seed


In [ ]:
print("Phase 4 BTUMQA QA-PRUGTM hybrid adaptive uncertainty output files by seed:")

for run_seed in RUN_SEEDS:
    activate_seed_context(run_seed)
    print("\n" + "=" * 100)
    print(f"{SEED_TAG} output files:")
    print("Run directory:", PHASE4_DIR)

    if PHASE4_DIR.exists():
        for path in sorted(PHASE4_DIR.rglob("*")):
            if path.is_file():
                size_mb = path.stat().st_size / (1024 ** 2)
                print(f"{str(path.relative_to(PHASE4_DIR)):60s} {size_mb:9.2f} MB")
    else:
        print("Run directory does not exist yet.")

    print("\nBest checkpoint exists:", BEST_CKPT_PATH.exists(), BEST_CKPT_PATH)
    print("Last checkpoint exists:", LAST_CKPT_PATH.exists(), LAST_CKPT_PATH)
    print("Val predictions exists:", VAL_PREDICTIONS_PATH.exists(), VAL_PREDICTIONS_PATH)
    print("Test predictions exists:", TEST_PREDICTIONS_PATH.exists(), TEST_PREDICTIONS_PATH)
    print("Final report exists:", PHASE4_REPORT_PATH.exists(), PHASE4_REPORT_PATH)
    print("Val query context summary exists:", VAL_QUERY_CONTEXT_PATH.exists(), VAL_QUERY_CONTEXT_PATH)
    print("Test query context summary exists:", TEST_QUERY_CONTEXT_PATH.exists(), TEST_QUERY_CONTEXT_PATH)
    print("Val post reasoning summary exists:", VAL_POST_REASONING_PATH.exists(), VAL_POST_REASONING_PATH)
    print("Test post reasoning summary exists:", TEST_POST_REASONING_PATH.exists(), TEST_POST_REASONING_PATH)

    report_preview = read_json(PHASE4_REPORT_PATH, {})
    if report_preview:
        print("\nModel variant:", report_preview.get("model_variant"))
        print("Seed tag:", report_preview.get("seed_tag"))
        print("Best epoch:", report_preview["training"]["best_epoch"] + 1)
        print("Best val macro-F1:", report_preview["training"]["best_val_macro_f1"])
        print("Val accuracy:", report_preview["validation"]["accuracy"])
        print("Test accuracy:", report_preview["test"]["accuracy"])
        print("Test macro-F1:", report_preview["test"]["macro_f1"])


Phase 4 BTUMQA QA-PRUGTM hybrid adaptive uncertainty output files by seed:

seed_42 output files:
Run directory: /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_4_qgca_training/biasless_metadata_free_training/adaptive_prugtm_qgca_btumqa_four_seeds/seed_42
checkpoints/best_adaptive_prugtm_qgca_model.pt                   44.34 MB
checkpoints/last_adaptive_prugtm_qgca_model.pt                   44.34 MB
done/phase4_qa_prugtm_hybrid_final_eval_complete.json             0.00 MB
done/phase4_qa_prugtm_hybrid_training_complete.json               0.00 MB
phase4_qa_prugtm_hybrid_config.json                               0.01 MB
phase4_qa_prugtm_hybrid_training_history.json                     0.43 MB
phase4_qa_prugtm_hybrid_training_report.json                      0.07 MB
predictions/test_predictions.csv                                  5.96 MB
predictions/val_predictions.csv                                   5.93 MB
reports/final_eval